In [1]:
# CELULA 01
# ============================================================================
# SETUP & IMPORTS 
# ============================================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuração de exibição do Pandas
pd.options.display.float_format = '{:,.2f}'.format

def formatar_moeda(valor):
    if isinstance(valor, (int, float)):
        return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return valor

def formatar_pct(valor):
    return f"{valor:.1f}%"

print("✅ Ambiente configurado")
print("📦 Bibliotecas: pandas, numpy, plotly, scipy")
print("🎨 Cores: Movidas para PREMISSAS (Célula 02)")

✅ Ambiente configurado
📦 Bibliotecas: pandas, numpy, plotly, scipy
🎨 Cores: Movidas para PREMISSAS (Célula 02)


In [2]:
# CÉLULA 02 - PREMISSAS V9.0 - PAINEL DE CONTROLE MASTER (HUMAN READABLE)
# ============================================================================
# DOCUMENTAÇÃO E ESTRATÉGIA
# ============================================================================
# CENÁRIO: BOOTSTRAPPING (Validação com Recursos Próprios)
#
# REGRAS FINANCEIRAS:
# 1. Caixa Inicial: R$ 8.000,00 (Aporte D0)
# 2. Aporte Mensal: R$ 2.000,00 (Teto de gasto do investidor)
# 3. Marketing: R$ 2.000,00 (Consome 100% do aporte, infra paga com receita)
# 4. CAPEX: R$ 0,00 (Hardware/PC assumido na Pessoa Física)
#
# REGRAS DE GROWTH:
# 1. Metas são definidas MENSALMENTE.
# 2. A taxa SEMANAL é calculada matematicamente no final desta célula.
# ============================================================================

PREMISSAS = {
    # ========================================================================
    # 1. METADADOS & CONTROLE DE TEMPO
    # ========================================================================
    'meta_version': '9.0-full-documented',
    'meta_updated_at': '2025-12-01',
    
    'data_inicio': '2025-11-01',
    'meses_projecao': 36,       # 3 anos de visão
    'seed_fixa': 42,            # Para reprodutibilidade (Monte Carlo)
    
    # Convenções de Tempo
    'semanas_por_mes': 4.33,
    'meses_por_ano': 12,

    # ========================================================================
    # 2. SAZONALIDADE DO MERCADO (CALENDÁRIO B3)
    # ========================================================================
    # Fator multiplicador sobre o tráfego base.
    # Jan (1.30) é forte (começo de ano), Dez (0.75) é fraco (festas).
    'sazonalidade_trafego': {
        1: 1.30,  2: 1.10,  3: 1.15, 
        4: 1.00,  5: 1.05,  6: 1.00,
        7: 0.85,  8: 1.00,  9: 1.00, 
        10: 1.05, 11: 1.10, 12: 0.75
    },

    # ========================================================================
    # 3. AQUISIÇÃO, GROWTH & METAS
    # ========================================================================
    'usuarios_pagos_iniciais': 5,  # Começamos pequenos
    'trafego_inicial': 500,        # Visitas no site/app
    
    # --- METAS DE CRESCIMENTO MENSAL (DRIVERS) ---
    # A meta semanal será calculada baseada nestes números no final do script.
    'crescimento_trafego_mes_1_6': 0.50,      # 15% ao mês (Fase Validação)
    'crescimento_trafego_mes_7_12': 0.06,     # 6% ao mês (Fase Consolidação)
    'crescimento_trafego_mes_13_plus': 0.03,  # 3% ao mês (Fase Escala)
    
    # Configuração da Validação Semanal
    'semanas_validacao_m0_m6': 26,            # Duração da fase crítica (6 meses)

    # --- FUNIL DE VENDAS ---
    'taxa_visitante_para_trial': 0.03,        # 3% dos visitantes testam
    'taxa_trial_para_pagante': 0.10,          # 10% dos testers pagam
    
    # Eficiência do Time (Ramp-up)
    'ramp_up_inicial': 0.50,                  # Começamos com 50% de eficiência
    'ramp_up_incremento': 0.06,               # Melhora 6% ao mês até 100%
    
    # --- CRESCIMENTO ORGÂNICO (SEO/INDICAÇÃO) ---
    'fator_visitas_organicas_por_pagante': 3.0, # Cada cliente atrai 3 visitantes
    'taxa_crescimento_organico_base': 0.02,     # SEO melhora 2% ao mês
    'elasticidade_organico': 0.60,              # Fator de viralidade
    'alerta_dependencia_organica_pct': 50.0,    # Alerta se depender <50% de orgânico

    # ========================================================================
    # 4. CHURN & RETENÇÃO (CANCELAMENTO)
    # ========================================================================
    # Traders trocam muito de plataforma no início.
    'churn_inicial': 0.12,                    # 12% ao mês (Mês 1)
    'churn_maturidade': 0.05,                 # 5% ao mês (Meta Longo Prazo)
    'churn_base': 0.08,                       # Média para cálculos simples
    'churn_decaimento_mensal': 0.0015,        # Melhora 0.15% a cada mês
    'taxa_reativacao_base_cancelada': 0.01,   # 1% dos cancelados voltam

    # ========================================================================
    # 5. PRICING & MIX DE PRODUTOS
    # ========================================================================
    # Preços das Assinaturas
    'preco_lite': 69.90, 
    'preco_trader': 99.90, 
    'preco_pro': 169.90,
    
    # Distribuição dos Clientes (Mix)
    'mix_lite': 0.50,                         # 50% no plano básico
    'mix_trader': 0.35,                       # 35% no intermediário
    'mix_pro': 0.15,                          # 15% no avançado
    'taxa_upgrade_lite_trader': 0.02,         # 2% sobem de plano/mês
    
    # --- MEIOS DE PAGAMENTO ---
    'mix_pagamento_cartao': 0.55, 
    'mix_pagamento_pix': 0.40, 
    'mix_pagamento_boleto': 0.05,
    
    # Custos Financeiros (Taxas)
    'taxa_processamento_cartao': 0.045,       # 4.5%
    'taxa_processamento_pix': 0.01,           # 1.0%
    'taxa_processamento_boleto': 0.035,       # 3.5%
    'taxa_chargeback': 0.005,                 # 0.5% fraude/estorno
    
    # Inadimplência Realista
    'taxa_inadimplencia_cartao': 0.04,
    'taxa_inadimplencia_pix': 0.08,           # Pix agendado que não paga
    'taxa_inadimplencia_boleto': 0.02,
    
    # Regime Tributário
    'imposto_simples_inicial': 0.06,          # 6% (Anexo III)
    'imposto_lucro_presumido': 0.1633,        # 16.33% (Se estourar o teto)
    'threshold_regime_tributario': 4800000.00,# Teto Simples Nacional

    # ========================================================================
    # 6. INFRAESTRUTURA & CUSTOS VARIÁVEIS (TIERS)
    # ========================================================================
    # Custo Variável por Usuário (IA Tokens + Server Load)
    'custo_ia_lite': 2.50, 
    'custo_ia_trader': 4.50, 
    'custo_ia_pro': 13.00,
    
    'custo_ferramentas_base': 0,         # Ferramentas fixas (Jira, etc), já está nos custos de ferramentas do tier 1!!!
    'custo_suporte_por_1000_users': 1.00,   # Zendesk variável - não tem agoagora eu o fundador vou dar suporte e vou automatizar!!!
    
    # Limites para Mudança de Tier (Escalabilidade)
    'infra_tier_1_limite': 500,
    'infra_tier_2_limite': 1500,
    'infra_tier_3_limite': 5000,
    'infra_tier_4_limite': 10000,

    # --- TIER 1: VALIDAÇÃO (0 - 500 usuários) ---
    't1_vps_app_api': 120.00, 
    't1_vps_windows_mt5': 180.00, 
    't1_database_managed': 80.00,
    't1_storage_s3': 20.00, 
    't1_ferramentas_dev': 300.00, 
    't1_observability': 0.00,
    't1_scraping_news': 200.00, 
    't1_dominio_dns': 10.00, 
    't1_email_transacional': 50.00,
    
    # --- TIER 2: GROWTH (501 - 1500 usuários) ---
    't2_api_dados_b3': 5000.00, 
    't2_compute_app': 450.00, 
    't2_database_primary': 350.00,
    't2_cache_redis': 120.00, 
    't2_storage_s3': 100.00, 
    't2_security_waf': 120.00,
    't2_ferramentas_dev': 600.00, 
    't2_observability': 250.00, 
    't2_suporte_ticket': 300.00,
    
    # --- TIER 3: SCALE (1501 - 5000 usuários) ---
    't3_api_dados_b3_pro': 7000.00, 
    't3_load_balancer': 200.00, 
    't3_compute_cluster': 1500.00,
    't3_db_primary_replica': 1200.00, 
    't3_cache_cluster': 400.00, 
    't3_data_warehouse': 500.00,
    't3_security_advanced': 500.00, 
    't3_ci_cd_pipeline': 300.00, 
    't3_observability_pro': 1000.00,
    
    # --- TIER 4: ENTERPRISE (5001+ usuários) ---
    't4_api_dados_institutional': 10000.00, 
    't4_k8s_cluster': 4500.00,
    't4_db_aurora_serverless': 3000.00, 
    't4_data_lake_engineering': 2000.00,
    't4_security_soc': 2500.00, 
    't4_support_enterprise': 1500.00, 
    't4_multi_region_backup': 1000.00,

    # ========================================================================
    # 7. MARKETING (REALIDADE BOOTSTRAPPING)
    # ========================================================================
    # R$ 2.000 é o teto máximo.
    # O script de validação vai checar se isso paga o CAC.
    'marketing_fixo_mensal': 2000.00,
    'marketing_perc_receita': 0.40,           # Reinveste 40% da receita em ads
    'marketing_teto': 25000.00,               # Teto futuro
    
    # Mix de Canais (Onde gastamos o dinheiro)
    'canal_instagram_pct': 0.20, 'cpc_instagram': 0.60, 'conv_instagram': 0.03,
    'canal_facebook_pct': 0.20,  'cpc_facebook': 0.80,  'conv_facebook': 0.02,
    'canal_youtube_pct': 0.30,   'cpc_youtube': 3.00,   'conv_youtube': 0.09,
    'canal_google_pct': 0.30,    'cpc_google': 5.00,    'conv_google': 0.12,

    # ========================================================================
    # 8. PROGRAMA DE AFILIADOS
    # ========================================================================
    'modelo_afiliado_habilitado': True,
    'pct_usuarios_via_afiliado': 0.30,        # 30% das vendas vêm daqui
    'comissao_afiliado_tipo': 'primeira_mensalidade', # Paga só a 1ª
    'comissao_afiliado_fixo': 100.00, 
    'comissao_afiliado_pct': 0.20, 
    'comissao_afiliado_meses': 12,

    # ========================================================================
    # 9. RH & EQUIPE (GATILHOS DE CONTRATAÇÃO)
    # ========================================================================
    # Só contrata se atingir gatilhos de Receita ou Usuários
    'salario_fundador': 5000.00,    'trigger_fundador': 15000.00, # MRR > 15k
    'salario_dev_senior': 10000.00, 'trigger_dev': 750,           # Users > 750
    'salario_cs': 4500.00,          'trigger_cs': 1000,           # Users > 1000
    'encargos_trabalhistas': 0.70,  # CLT + Benefícios

    # ========================================================================
    # 10. GOVERNANÇA, ADMIN & B2B
    # ========================================================================
    'custo_escritorio_base': 3500.00, 
    'trigger_escritorio': 65000.00,           # Só aluga sala se faturar 65k
    
    'custo_contabilidade_adv': 1000.00,       # Contador
    'custo_juridico_compliance': 100.00,
    'trigger_contabilidade': 65000.00,      #adicionado agora, precisa corrigir motor e outras celular
    
    'verba_viagens_base': 1.00, 
    'verba_viagens_pct_receita': 0.02,
    'trigger_viagens_receita': 35000.00,
    
    'conselho_jeton': 3000.00, 
    'trigger_conselho_mrr': 100000.00,
    
    'freelancers_trimestral': 3000.00,
    'trigger_freelancer_receita': 30000.00,
    
    'beneficios_executivos': 4500.00, 
    'trigger_beneficios_lucro': 80000.00, 
    
    # B2B (Venda para Escolas/Mesas)
    'b2b_probabilidade_anual': 0.30, 
    'b2b_setup_fee': 15000.00,
    'b2b_custo_implantacao': 5000.00,

    # ========================================================================
    # 11. CAPITAL & CONTABILIDADE (REALIDADE DO CLIENTE)
    # ========================================================================
    'caixa_inicial': 0.00,       # Dinheiro na conta hoje
    'aporte_mensal': 2000.00,       # Quanto entra por mês
    'meses_aporte': 10,             # Garantia de aporte por 6 meses, não mexe nisso!!!
    
    'capex_inicial': 8000.00,          # PC comprado na PF (Zero custo empresa)
    'capex_recorrente_24m': 12000.00,     
    'tempo_depreciacao_equipamento_meses': 24,

    # ========================================================================
    # 12. DISTRIBUIÇÃO DE LUCROS
    # ========================================================================
    'percentual_distribuicao': 0.00, # 0% - Reinvestimento total na fase inicial
    'split_fundador': 0.50,
    'split_investidor': 0.50,

    # ========================================================================
    # 13. BENCHMARKS & TRADUÇÕES (OBRIGATÓRIOS)
    # ========================================================================
    # Benchmarks Fintech B2C (Validação M0-M6)
    'benchmark_cac_m0': 400.0,
    'benchmark_cac_m6': 200.0,
    'benchmark_cac_m12': 180.0,
    'benchmark_cac_m36': 150.0,
    
    'benchmark_churn_m0': 0.12,
    'benchmark_churn_m6': 0.075,
    
    'benchmark_ltv_cac_m0': 2.0,
    'benchmark_ltv_cac_m6': 4.0,

    # Benchmarks para Validação Final (Status Report)
    'benchmark_churn_atencao_pct': 5.0,
    'benchmark_churn_critico_pct': 7.0,
    'benchmark_margem_alvo_pct': 80.0,
    'benchmark_margem_critica_pct': 70.0,
    'benchmark_ltv_cac_excelente': 5.0,
    'benchmark_ltv_cac_atencao': 3.0,
    'benchmark_ltv_cac_critico': 1.0,
    'benchmark_ltv_cac_suspeito': 10.0,
    
    # Traduções Obrigatórias (Zero Jargão)
    'traducoes': {
        'Runway': 'Runway (Sobrevivência / Pista de Pouso)',
        'Burn Rate': 'Burn Rate (Taxa de Queima / Queima de Caixa)',
        'Churn': 'Churn (Cancelamento / Evasão)',
        'LTV': 'LTV (Valor Vitalício do Cliente)',
        'CAC': 'CAC (Custo de Aquisição)',
        'Payback': 'Payback (Tempo de Retorno)',
        'Margem Contrib': 'Margem de Contribuição',
        'MRR': 'MRR (Receita Recorrente Mensal)',
        'ARR': 'ARR (Receita Anual Recorrente)',
        'ARPU': 'ARPU (Receita Média por Usuário)',
    },

    # ========================================================================
    # 14. METAS DO NEGÓCIO (OBJETIVOS)
    # ========================================================================
    'meta_caixa_seguranca': 50000.0,
    'meta_caixa_ideal': 100000.0,
    'meta_runway_minimo_meses': 6,

    # ========================================================================
    # 15. MONTE CARLO (SIMULAÇÃO DE RISCO)
    # ========================================================================
    'mc_n_simulacoes': 100,
    'mc_prob_atraso_aporte': 0.20,
    'mc_std_churn': 0.30,
    'mc_std_vis_trial': 0.30,
    'mc_std_trial_pag': 0.30,
    'mc_std_cresc_traf': 0.35,
    'mc_std_marketing': 0.15,
    'mc_std_infra': 0.15,
    'mc_std_taxa_pag': 0.005,
    'mc_std_mix_lite': 0.10,
    'mc_var_ia_otimista': 0.80,
    'mc_var_ia_pessimista': 1.50,
    
    # ========================================================================
    # 16. DESIGN SYSTEM (CORES COMPLETAS)
    # ========================================================================
    'cores': {
        'primary': '#0F172A',    # Navy Blue (Principal)
        'secondary': '#64748B',  # Slate (Secundário)
        'background': '#FFFFFF', # Branco (Fundo)
        'grid': '#F1F5F9',       # Cinza Claro (Linhas)
        
        # Cores Semânticas
        'receita': '#10B981',    # Verde (Entrada)
        'despesa': '#EF4444',    # Vermelho (Saída)
        'lucro': '#3B82F6',      # Azul (Resultado)
        'ebitda': '#6366F1',     # Indigo (Operacional)
        'caixa': '#8B5CF6',      # Roxo (Acumulado)
        
        # Status
        'success': '#10B981',    # Sucesso
        'danger': '#EF4444',     # Perigo
        'warning': '#F59E0B',    # Atenção
        'info': '#3B82F6',       # Informativo
        
        # Categorias
        'marketing_pago': '#3B82F6',
        'marketing_organico': '#10B981',
        'infra': '#64748B',
        'cogs': '#E05D44',
        'opex': '#2684FF'
    }
}

# ============================================================================
# CÁLCULOS AUTOMÁTICOS (PÓS-CONFIGURAÇÃO)
# ============================================================================
# Aqui transformamos a meta mensal em meta semanal matematicamente.
# Não há "outra meta", é a MESMA meta, convertida.

taxa_mensal_meta = PREMISSAS['crescimento_trafego_mes_1_6']
semanas_no_mes = PREMISSAS['semanas_por_mes']

# Fórmula de juros compostos: (1 + Mensal) = (1 + Semanal) ^ Semanas
# Logo: Semanal = (1 + Mensal)^(1/Semanas) - 1
taxa_semanal_calc = (1 + taxa_mensal_meta) ** (1 / semanas_no_mes) - 1

# Atualiza no dicionário
PREMISSAS['taxa_crescimento_semanal_m0_m6'] = taxa_semanal_calc

print("="*80)
print("✅ PREMISSAS V9.0 CARREGADAS E CALCULADAS")
print("="*80)
print(f"💰 RESUMO FINANCEIRO:")
print(f"   • Caixa Inicial:        R$ {PREMISSAS['caixa_inicial']:,.2f}")
print(f"   • Aporte Mensal:        R$ {PREMISSAS['aporte_mensal']:,.2f}")
print(f"   • Budget Mkt Mensal:    R$ {PREMISSAS['marketing_fixo_mensal']:,.2f}")

print(f"\n📈 METAS CONVERTIDAS:")
print(f"   • Meta Mensal Definida: {taxa_mensal_meta*100:.1f}%")
print(f"   • Meta Semanal Calc.:   {taxa_semanal_calc*100:.2f}% (Matematicamente equivalente)")

print(f"\n🎨 CORES CARREGADAS: {len(PREMISSAS['cores'])} definições.")
print("⭐ PRONTO PARA EXECUÇÃO.")

✅ PREMISSAS V9.0 CARREGADAS E CALCULADAS
💰 RESUMO FINANCEIRO:
   • Caixa Inicial:        R$ 0.00
   • Aporte Mensal:        R$ 2,000.00
   • Budget Mkt Mensal:    R$ 2,000.00

📈 METAS CONVERTIDAS:
   • Meta Mensal Definida: 50.0%
   • Meta Semanal Calc.:   9.82% (Matematicamente equivalente)

🎨 CORES CARREGADAS: 18 definições.
⭐ PRONTO PARA EXECUÇÃO.


In [3]:
# CÉLULA 03 - DIAGNÓSTICO DE VIABILIDADE (V4.0 - MAIS ROBUSTO)
# ============================================================================
# OBJETIVO: Calcular a coerência matemática das premissas de forma mais precisa.
# MELHORIAS:
# 1. Usa o funil de marketing para projetar aquisição realista.
# 2. Calcula a margem bruta de forma dinâmica, sem números mágicos.
# ============================================================================

def diagnosticar_viabilidade_refatorado(premissas):
    """
    Analisa se as metas cabem no budget e se os unit economics param de pé.
    Gera um RELATÓRIO DE DIAGNÓSTICO mais preciso.
    """
    import numpy as np
    
    # 1. Setup de Variáveis
    budget_mensal = premissas['marketing_fixo_mensal'] # R$ 2.000
    
    # CAC Estimado (Benchmark do mercado)
    cac_estimado_benchmark = premissas['benchmark_cac_m0'] # R$ 400

    # --- MELHORIA 1: Projetando aquisição a partir do funil e budget ---
    # Quantos clientes o budget permite adquirir com o CAC benchmark?
    usuarios_adquiriveis_com_budget = budget_mensal / cac_estimado_benchmark
    
    # Agora, qual CAC seria necessário para atingir uma meta de crescimento?
    # Vamos projetar uma meta de novos clientes para o Mês 1 usando o funil
    # (Este é um cálculo de projeção, o motor fará isso mensalmente)
    meta_trafego_m1 = premissas['trafego_inicial'] * (1 + premissas['crescimento_trafego_mes_1_6'])
    meta_trials_m1 = meta_trafego_m1 * premissas['taxa_visitante_para_trial']
    meta_pagantes_m1 = meta_trials_m1 * premissas['taxa_trial_para_pagante']
    
    cac_necessario_para_meta = budget_mensal / meta_pagantes_m1 if meta_pagantes_m1 > 0 else float('inf')

    # --- MELHORIA 2: Calculando a margem e LTV de forma dinâmica ---
    arpu = (premissas['preco_lite'] * premissas['mix_lite'] +
            premissas['preco_trader'] * premissas['mix_trader'] +
            premissas['preco_pro'] * premissas['mix_pro'])
    
    custo_var_medio = (premissas['custo_ia_lite'] * premissas['mix_lite'] +
                       premissas['custo_ia_trader'] * premissas['mix_trader'] +
                       premissas['custo_ia_pro'] * premissas['mix_pro'])
    
    margem_bruta_decimal = (arpu - custo_var_medio) / arpu
    
    ltv_estimado = (arpu * margem_bruta_decimal) / premissas['churn_base']
    ltv_cac = ltv_estimado / cac_estimado_benchmark

    print("="*80)
    print("🩺 DIAGNÓSTICO MATEMÁTICO REFINADO")
    print("="*80)

    # ANÁLISE 1: BUDGET vs. META (Versão Aprimorada)
    print(f"\n1️⃣  TESTE DE STRESS DO BUDGET (MÊS 1):")
    print(f"   • Budget Mensal Disp.:   R$ {budget_mensal:.2f}")
    print(f"   • CAC Benchmark:        R$ {cac_estimado_benchmark:.2f}")
    print(f"   • Capacidade de Aquisição (com CAC Benchmark): {usuarios_adquiriveis_com_budget:.1f} usuários/mês")
    print("-" * 40)
    print(f"   • Projeção de Novos Pagantes (Mês 1): {meta_pagantes_m1:.1f} usuários")
    print(f"   • CAC NECESSÁRIO para a meta:    R$ {cac_necessario_para_meta:.2f}")
    
    if cac_necessario_para_meta > cac_estimado_benchmark:
        print(f"   ⚠️  ALERTA: Para atingir sua meta de {meta_pagantes_m1:.1f} usuários, o CAC precisa ser de R$ {cac_necessario_para_meta:.2f}.")
        print(f"       Isso é MAIOR que o benchmark de mercado (R$ {cac_estimado_benchmark:.2f}).")
        print(f"       📝 INTERPRETAÇÃO: Sua meta de crescimento é muito agressiva para o budget inicial.")
        print(f"       Você precisa: (A) Aumentar o budget, (B) Melhorar a conversão do funil, ou (C) Aceitar um crescimento mais lento.")
    else:
        print(f"   ✅ OK: O budget parece suficiente para atingir a meta de crescimento projetada.")

    # ANÁLISE 2: ECONOMIA DO CLIENTE (Versão Dinâmica)
    print(f"\n2️⃣  ECONOMIA DO CLIENTE (Unit Economics):")
    print(f"   • Receita Média (ARPU):           R$ {arpu:.2f}")
    print(f"   • Custo Variável Médio (IA):      R$ {custo_var_medio:.2f}")
    print(f"   • Margem Bruta (Dinâmica):        {margem_bruta_decimal:.1%}")
    print(f"   • LTV Estimado:                   R$ {ltv_estimado:.2f}")
    print(f"   • LTV/CAC (com Benchmark CAC):    {ltv_cac:.2f}x")
    
    if ltv_cac < 1:
        print("   🔴 CRÍTICO: Você gasta mais para trazer o cliente do que ele te paga (LTV < CAC).")
    elif ltv_cac < 3:
        print("   ⚠️  ATENÇÃO: LTV/CAC baixo (Normal no início, mas exige foco em retenção e otimização de CAC).")
    else:
        print("   ✅ SAUDÁVEL: A economia do cliente é positiva no longo prazo.")

    print("\n" + "="*80)
    print("🏁 DIAGNÓSTICO CONCLUÍDO. LIBERANDO MOTOR FINANCEIRO...")
    print("="*80)

# Executar a versão refatorada
diagnosticar_viabilidade_refatorado(PREMISSAS)

🩺 DIAGNÓSTICO MATEMÁTICO REFINADO

1️⃣  TESTE DE STRESS DO BUDGET (MÊS 1):
   • Budget Mensal Disp.:   R$ 2000.00
   • CAC Benchmark:        R$ 400.00
   • Capacidade de Aquisição (com CAC Benchmark): 5.0 usuários/mês
----------------------------------------
   • Projeção de Novos Pagantes (Mês 1): 2.2 usuários
   • CAC NECESSÁRIO para a meta:    R$ 888.89
   ⚠️  ALERTA: Para atingir sua meta de 2.2 usuários, o CAC precisa ser de R$ 888.89.
       Isso é MAIOR que o benchmark de mercado (R$ 400.00).
       📝 INTERPRETAÇÃO: Sua meta de crescimento é muito agressiva para o budget inicial.
       Você precisa: (A) Aumentar o budget, (B) Melhorar a conversão do funil, ou (C) Aceitar um crescimento mais lento.

2️⃣  ECONOMIA DO CLIENTE (Unit Economics):
   • Receita Média (ARPU):           R$ 95.40
   • Custo Variável Médio (IA):      R$ 4.78
   • Margem Bruta (Dinâmica):        95.0%
   • LTV Estimado:                   R$ 1132.81
   • LTV/CAC (com Benchmark CAC):    2.83x
   ⚠️  ATENÇÃO: 

In [4]:
# CÉLULA 04 - MOTOR FINANCEIRO V9.4 (PATCHED FINAL) - compatível com PREMISSAS
# Correções aplicadas: mix proporcional (inicial + mensal), RNG B2B independente,
# comentário de CAPEX corrigido, prevenção duplicação custo_ferramentas_base.
# Mantive a arquitetura, nomenclatura e vetores tal como V9.4 original.
# ====================================================================
def executar_motor_fintech_v7_investor_grade(p, seed=None, variacao_params=None, modo_debug=False):
    import pandas as pd
    import numpy as np

    # --------------------------------------------------------------------
    # BLOCO 0: CONFIGURAÇÃO E SETUP
    # --------------------------------------------------------------------
    p_run = p.copy()
    if variacao_params:
        for k, v in variacao_params.items():
            p_run[k] = v

    # Seed principal (reprodutibilidade)
    if seed is None:
        seed = p_run.get('seed_fixa', 42)
    seed = int(seed)
    np.random.seed(seed)

    meses = int(p_run['meses_projecao'])
    datas = pd.date_range(start=p_run['data_inicio'], periods=meses, freq='MS')
    alertas = []

    # --------------------------------------------------------------------
    # BLOCO 1: INICIALIZAÇÃO DE VETORES
    # --------------------------------------------------------------------
    dados = {}
    metricas_vetores = [
        'trafego_total', 'trafego_pago', 'trafego_organico',
        'trials_total', 'trials_pagos', 'trials_organicos',
        'novos_pagantes_total', 'novos_ads', 'novos_organicos', 'novos_afiliados',
        'reativacoes', 'taxa_trafego_trial', 'taxa_trial_pago', 'eficiencia_time',
        'usuarios_ativos', 'usuarios_lite', 'usuarios_trader', 'usuarios_pro',
        'churn_usuarios', 'churn_mrr', 'upgrades_lite_trader',
        'receita_bruta', 'receita_assinaturas', 'receita_b2b',
        'receita_lite', 'receita_trader', 'receita_pro',
        'mrr', 'arr', 'arpu', 'crescimento_mrr_mom', 'crescimento_mrr_yoy', 'net_new_mrr',
        'impostos', 'aliquota_efetiva', 'taxas_pagamento',
        'inadimplencia', 'chargeback', 'total_deducoes', 'receita_liquida',
        'custo_ia_lite', 'custo_ia_trader', 'custo_ia_pro', 'custo_ia_total',
        'comissao_afiliados', 'custo_suporte_variavel', 'total_cogs',
        'margem_bruta', 'margem_bruta_pct',
        'custo_infra_fixo', 'infra_tier_ativo',
        'headcount_total', 'headcount_fundadores', 'headcount_dev', 'headcount_cs',
        'custo_pessoal', 'salarios_brutos', 'encargos',
        'gasto_marketing', 'gasto_instagram', 'gasto_facebook', 'gasto_youtube', 'gasto_google',
        'cpc_blended', 'custo_escritorio', 'custo_contabilidade',
        'despesas_viagens', 'despesas_conselho', 'despesas_freelancer', 'despesas_beneficios',
        'total_opex',
        'margem_contribuicao', 'margem_contribuicao_pct',
        'ebitda', 'ebitda_margin',
        'depreciacao', 'ebit', 'ebit_margin',
        'lucro_liquido', 'margem_liquida',
        'fluxo_operacional', 'fluxo_investimento', 'fluxo_financiamento',
        'aportes_capital', 'capex', 'distribuicao_lucros',
        'caixa', 'burn_rate', 'runway_meses',
        'ltv', 'cac_blended', 'cac_paid', 'ltv_cac',
        'payback_meses', 'payback_semanas', 'vida_media_cliente',
        'churn_rate', 'retention_rate', 'regra_40', 'burn_multiple'
    ]
    for m in metricas_vetores:
        dados[m] = np.zeros(meses)

    # --------------------------------------------------------------------
    # BLOCO 2: ESTADO INICIAL (CORRIGIDO)
    # --------------------------------------------------------------------
    usuarios_ativos = p_run['usuarios_pagos_iniciais']

    # Mix inicial: calculo integral e distribuição proporcional do remainder
    base_lite = int(usuarios_ativos * p_run['mix_lite'])
    base_trader = int(usuarios_ativos * p_run['mix_trader'])
    base_pro = int(usuarios_ativos * p_run['mix_pro'])
    diff_inicial = int(usuarios_ativos) - (base_lite + base_trader + base_pro)

    # CORREÇÃO APLICADA: distribuição proporcional do diff inicial conforme mix
    if diff_inicial != 0:
        mixes = np.array([p_run['mix_lite'], p_run['mix_trader'], p_run['mix_pro']], dtype=float)
        total_mix = mixes.sum()
        if total_mix <= 0:
            # fallback: joga tudo no lite
            base_lite += diff_inicial
        else:
            proporcoes = mixes / total_mix
            ajustes = np.floor(proporcoes * diff_inicial).astype(int)
            base_lite += ajustes[0]
            base_trader += ajustes[1]
            base_pro += ajustes[2]
            restante = diff_inicial - ajustes.sum()
            # distribui o restante para os planos com maior proporção (fair rounding)
            if restante > 0:
                idx_ord = np.argsort(-proporcoes)  # índices de maior para menor
                for r in range(restante):
                    if idx_ord[r % 3] == 0:
                        base_lite += 1
                    elif idx_ord[r % 3] == 1:
                        base_trader += 1
                    else:
                        base_pro += 1
            elif restante < 0:
                # caso negativo raro, remove do maior mix
                idx_ord = np.argsort(-proporcoes)
                for r in range(abs(restante)):
                    if idx_ord[r % 3] == 0 and base_lite > 0:
                        base_lite -= 1
                    elif idx_ord[r % 3] == 1 and base_trader > 0:
                        base_trader -= 1
                    elif base_pro > 0:
                        base_pro -= 1

    # CORREÇÃO CRÍTICA: Não subtrai CAPEX aqui para evitar duplicação.
    # O CAPEX será descontado apenas no fluxo do Mês 0 (Bloco 3.10).
    caixa_atual = p_run['caixa_inicial']

    trafego_base = p_run['trafego_inicial']
    pool_churned_users = 0
    flags = {'fundador': False, 'dev': False, 'cs': False}

    # --------------------------------------------------------------------
    # BLOCO 3: LOOP MENSAL
    # --------------------------------------------------------------------
    # Para B2B com aleatoriedade não-viciada, criamos RNG secundário derivado da seed
    # Observação: usamos np.random.seed(seed) acima para reprodutibilidade global.
    for mes in range(meses):
        mes_atual_num = mes + 1
        mes_cal = datas[mes].month

        # --------------------------
        # 3.1 GROWTH (FASES + ELASTICIDADE)
        # --------------------------
        if mes_atual_num <= 6:
            taxa_crescimento_atual = p_run['crescimento_trafego_mes_1_6']
        elif mes_atual_num <= 12:
            taxa_crescimento_atual = p_run['crescimento_trafego_mes_7_12']
        else:
            taxa_crescimento_atual = p_run['crescimento_trafego_mes_13_plus']

        if mes > 0:
            trafego_base *= (1 + taxa_crescimento_atual)
        fator_sazon = p_run['sazonalidade_trafego'].get(mes_cal, 1.0)

        if mes == 0:
            budget_mkt = p_run['marketing_fixo_mensal']
        else:
            budget_calc = dados['receita_bruta'][mes - 1] * p_run['marketing_perc_receita']
            budget_mkt = np.clip(budget_calc, p_run['marketing_fixo_mensal'], p_run['marketing_teto'])

        dados['gasto_marketing'][mes] = budget_mkt

        cpc_blended = (
            p_run['cpc_instagram'] * p_run['canal_instagram_pct'] +
            p_run['cpc_facebook'] * p_run['canal_facebook_pct'] +
            p_run['cpc_google'] * p_run['canal_google_pct'] +
            p_run['cpc_youtube'] * p_run['canal_youtube_pct']
        )
        dados['cpc_blended'][mes] = cpc_blended

        visitas_pagas = (budget_mkt / max(cpc_blended, 0.01)) * fator_sazon

        # Elasticidade Viral aplicada
        elasticidade = p_run.get('elasticidade_organico', 1.0)
        visitas_virais = usuarios_ativos * p_run.get('fator_visitas_organicas_por_pagante', 0.0) * elasticidade
        visitas_organicas = (trafego_base + visitas_virais) * fator_sazon

        dados['trafego_pago'][mes] = visitas_pagas
        dados['trafego_organico'][mes] = visitas_organicas
        dados['trafego_total'][mes] = visitas_pagas + visitas_organicas

        # --------------------------
        # 3.2 CONVERSÃO
        # --------------------------
        conv_blended = (
            p_run['conv_instagram'] * p_run['canal_instagram_pct'] +
            p_run['conv_facebook'] * p_run['canal_facebook_pct'] +
            p_run['conv_google'] * p_run['canal_google_pct'] +
            p_run['conv_youtube'] * p_run['canal_youtube_pct']
        )

        trials_pagos = visitas_pagas * conv_blended
        trials_organicos = visitas_organicas * p_run['taxa_visitante_para_trial']
        dados['trials_total'][mes] = trials_pagos + trials_organicos

        eficiencia = min(1.0, p_run['ramp_up_inicial'] + (mes * p_run['ramp_up_incremento']))
        dados['eficiencia_time'][mes] = eficiencia

        novos_bruto = dados['trials_total'][mes] * p_run['taxa_trial_para_pagante'] * eficiencia

        pct_afiliado = p_run['pct_usuarios_via_afiliado'] if p_run.get('modelo_afiliado_habilitado', False) else 0
        novos_afiliado = novos_bruto * pct_afiliado
        novos_resto = novos_bruto - novos_afiliado

        total_trials_attribution = trials_pagos + trials_organicos
        ratio_ads = (trials_pagos / total_trials_attribution) if total_trials_attribution > 0 else 0

        dados['novos_afiliados'][mes] = int(novos_afiliado)
        dados['novos_ads'][mes] = int(novos_resto * ratio_ads)
        dados['novos_organicos'][mes] = int(novos_resto * (1 - ratio_ads))
        dados['novos_pagantes_total'][mes] = (
            dados['novos_afiliados'][mes] + dados['novos_ads'][mes] + dados['novos_organicos'][mes]
        )

        # --------------------------
        # 3.3 CHURN
        # --------------------------
        reativacoes = int(pool_churned_users * p_run.get('taxa_reativacao_base_cancelada', 0.01)) if pool_churned_users > 0 else 0
        dados['reativacoes'][mes] = reativacoes

        churn_rate = max(p_run['churn_maturidade'], p_run['churn_inicial'] - (mes * p_run['churn_decaimento_mensal']))
        dados['churn_rate'][mes] = churn_rate

        churn_users = int(usuarios_ativos * churn_rate)
        dados['churn_usuarios'][mes] = churn_users
        pool_churned_users += churn_users - reativacoes

        usuarios_ativos = max(0, usuarios_ativos + dados['novos_pagantes_total'][mes] - churn_users + reativacoes)
        dados['usuarios_ativos'][mes] = int(usuarios_ativos)

        # --------------------------
        # 3.3.1 RECALCULO DO MIX (DISTRIBUIÇÃO PROPORCIONAL DO DIFF MENSAL)
        # --------------------------
        # Calcula base por mix e aplica upgrades
        upgrades = int(base_lite * p_run.get('taxa_upgrade_lite_trader', 0.0))
        # inicialmente distribui pela proporção desejada
        base_lite = int(usuarios_ativos * p_run['mix_lite']) - upgrades
        base_trader = int(usuarios_ativos * p_run['mix_trader']) + upgrades
        base_pro = int(usuarios_ativos * p_run['mix_pro'])

        diff = int(usuarios_ativos) - (base_lite + base_trader + base_pro)
        if diff != 0:
            mixes = np.array([p_run['mix_lite'], p_run['mix_trader'], p_run['mix_pro']], dtype=float)
            total_mix = mixes.sum()
            if total_mix <= 0:
                base_lite += diff
            else:
                proporcoes = mixes / total_mix
                ajustes = np.floor(proporcoes * diff).astype(int)
                base_lite += ajustes[0]
                base_trader += ajustes[1]
                base_pro += ajustes[2]
                restante = diff - ajustes.sum()
                if restante > 0:
                    idx_ord = np.argsort(-proporcoes)
                    for r in range(restante):
                        if idx_ord[r % 3] == 0:
                            base_lite += 1
                        elif idx_ord[r % 3] == 1:
                            base_trader += 1
                        else:
                            base_pro += 1
                elif restante < 0:
                    idx_ord = np.argsort(-proporcoes)
                    for r in range(abs(restante)):
                        if idx_ord[r % 3] == 0 and base_lite > 0:
                            base_lite -= 1
                        elif idx_ord[r % 3] == 1 and base_trader > 0:
                            base_trader -= 1
                        elif base_pro > 0:
                            base_pro -= 1

        dados['usuarios_lite'][mes] = base_lite
        dados['usuarios_trader'][mes] = base_trader
        dados['usuarios_pro'][mes] = base_pro

        # --------------------------
        # 3.4 RECEITA & DEDUÇÕES
        # --------------------------
        mrr = (base_lite * p_run['preco_lite'] + base_trader * p_run['preco_trader'] + base_pro * p_run['preco_pro'])
        dados['mrr'][mes] = mrr
        dados['arr'][mes] = mrr * 12
        dados['arpu'][mes] = mrr / usuarios_ativos if usuarios_ativos > 0 else 0

        # B2B com aleatoriedade independente (para não viciar Monte Carlo)
        # Usa RNG secundário derivado da seed principal + mês para variação
        rng_b2b = np.random.RandomState((seed + mes + 7) % (2**31 - 1))
        chance_b2b = rng_b2b.random_sample()
        rec_b2b = p_run['b2b_setup_fee'] if chance_b2b < (p_run.get('b2b_probabilidade_anual', 0) / 12) else 0
        dados['receita_bruta'][mes] = mrr + rec_b2b
        dados['receita_b2b'][mes] = rec_b2b

        if dados['receita_bruta'][mes] > 0:
            aliquota = p_run['imposto_lucro_presumido'] if dados['arr'][mes] > p_run['threshold_regime_tributario'] else p_run['imposto_simples_inicial']
            dados['impostos'][mes] = dados['receita_bruta'][mes] * aliquota

            taxa_inadimp = (
                p_run['taxa_inadimplencia_cartao'] * p_run['mix_pagamento_cartao'] +
                p_run['taxa_inadimplencia_pix'] * p_run['mix_pagamento_pix'] +
                p_run['taxa_inadimplencia_boleto'] * p_run['mix_pagamento_boleto']
            )
            dados['inadimplencia'][mes] = dados['receita_bruta'][mes] * taxa_inadimp

            taxa_pgto = (
                p_run['taxa_processamento_cartao'] * p_run['mix_pagamento_cartao'] +
                p_run['taxa_processamento_pix'] * p_run['mix_pagamento_pix'] +
                p_run['taxa_processamento_boleto'] * p_run['mix_pagamento_boleto']
            )
            dados['taxas_pagamento'][mes] = (dados['receita_bruta'][mes] - dados['inadimplencia'][mes]) * taxa_pgto
            dados['chargeback'][mes] = dados['receita_bruta'][mes] * p_run['taxa_chargeback']
            dados['total_deducoes'][mes] = dados['impostos'][mes] + dados['inadimplencia'][mes] + dados['taxas_pagamento'][mes] + dados['chargeback'][mes]
        else:
            # Zera deduções para clareza
            dados['impostos'][mes] = 0.0
            dados['inadimplencia'][mes] = 0.0
            dados['taxas_pagamento'][mes] = 0.0
            dados['chargeback'][mes] = 0.0
            dados['total_deducoes'][mes] = 0.0

        dados['receita_liquida'][mes] = dados['receita_bruta'][mes] - dados['total_deducoes'][mes]

        # --------------------------
        # 3.5 COGS
        # --------------------------
        custo_ia = (base_lite * p_run['custo_ia_lite'] + base_trader * p_run['custo_ia_trader'] + base_pro * p_run['custo_ia_pro'])
        dados['custo_ia_total'][mes] = custo_ia

        comissao = 0.0
        if p_run.get('modelo_afiliado_habilitado', False):
            if p_run.get('comissao_afiliado_tipo') == 'primeira_mensalidade':
                # Pagamento da primeira mensalidade ao afiliado
                comissao = dados['novos_afiliados'][mes] * dados['arpu'][mes]
            else:
                comissao = dados['novos_afiliados'][mes] * dados['arpu'][mes] * p_run.get('comissao_afiliado_pct', 0.2)
        dados['comissao_afiliados'][mes] = comissao

        if usuarios_ativos >= p_run['trigger_cs']:
            flags['cs'] = True

        if flags['cs']:
            dados['custo_suporte_variavel'][mes] = (usuarios_ativos / 1000) * p_run['custo_suporte_por_1000_users']
        else:
            dados['custo_suporte_variavel'][mes] = 0.0

        dados['total_cogs'][mes] = custo_ia + comissao + dados['custo_suporte_variavel'][mes]
        dados['margem_bruta'][mes] = dados['receita_liquida'][mes] - dados['total_cogs'][mes]
        dados['margem_bruta_pct'][mes] = (dados['margem_bruta'][mes] / dados['receita_bruta'][mes]) * 100 if dados['receita_bruta'][mes] > 0 else 0

        # --------------------------
        # 3.6 INFRA
        # --------------------------
        tier = 1
        if usuarios_ativos > p_run['infra_tier_1_limite']: tier = 2
        if usuarios_ativos > p_run['infra_tier_2_limite']: tier = 3
        if usuarios_ativos > p_run['infra_tier_3_limite']: tier = 4

        # soma somente parâmetros numéricos começando com t{tier}_
        custo_infra = sum([v for k, v in p_run.items() if k.startswith(f"t{tier}_") and isinstance(v, (int, float))])
        dados['infra_tier_ativo'][mes] = tier
        dados['custo_infra_fixo'][mes] = custo_infra

        # --------------------------
        # 3.7 RH
        # --------------------------
        custo_rh = 0.0
        headcount = 0
        if dados['mrr'][mes] >= p_run['trigger_fundador']:
            flags['fundador'] = True
        if usuarios_ativos >= p_run['trigger_dev']:
            flags['dev'] = True

        if flags['fundador']:
            custo_rh += p_run['salario_fundador'] * (1 + p_run.get('encargos_trabalhistas', 0.7))
            headcount += 1
        if flags['dev']:
            custo_rh += p_run['salario_dev_senior'] * (1 + p_run.get('encargos_trabalhistas', 0.7))
            headcount += 1
        if flags['cs']:
            custo_rh += p_run['salario_cs'] * (1 + p_run.get('encargos_trabalhistas', 0.7))
            headcount += 1

        dados['custo_pessoal'][mes] = custo_rh
        dados['headcount_total'][mes] = headcount

        # --------------------------
        # 3.8 OPEX
        # --------------------------
        opex_extra = 0.0
        if dados['mrr'][mes] >= p_run.get('trigger_escritorio', 65000):
            dados['custo_escritorio'][mes] = p_run['custo_escritorio_base']

        trigger_contabil = p_run.get('trigger_contabilidade', 65000)
        if dados['receita_bruta'][mes] >= trigger_contabil:
            dados['custo_contabilidade'][mes] = p_run['custo_contabilidade_adv'] + p_run.get('custo_juridico_compliance', 0)

        # Previne duplicação de custo de ferramentas: se o tier já inclui 't{tier}_ferramentas_dev' > 0, não some o custo_ferramentas_base
        ferramentas = p_run.get('custo_ferramentas_base', 0)
        tier_ferramentas_key = f"t{tier}_ferramentas_dev"
        if usuarios_ativos > 0 and ferramentas > 0:
            if p_run.get(tier_ferramentas_key, 0) <= 0:
                opex_extra += ferramentas
            else:
                # já incluso no tier => não adiciona para evitar duplicação
                pass

        if dados['receita_bruta'][mes] >= p_run.get('trigger_viagens_receita', 35000):
            dados['despesas_viagens'][mes] = p_run['verba_viagens_base']
        if dados['mrr'][mes] >= p_run.get('trigger_conselho_mrr', 100000):
            dados['despesas_conselho'][mes] = p_run.get('conselho_jeton', 0)

        dados['total_opex'][mes] = (
            dados['gasto_marketing'][mes] + dados['custo_infra_fixo'][mes] + dados['custo_pessoal'][mes] +
            dados['custo_escritorio'][mes] + dados['custo_contabilidade'][mes] + dados['despesas_viagens'][mes] +
            dados['despesas_conselho'][mes] + opex_extra
        )

        # --------------------------
        # 3.9 RESULTADO FINANCEIRO
        # --------------------------
        dados['ebitda'][mes] = dados['margem_bruta'][mes] - dados['total_opex'][mes]
        dados['ebitda_margin'][mes] = (dados['ebitda'][mes] / dados['receita_bruta'][mes]) * 100 if dados['receita_bruta'][mes] > 0 else 0

        tempo_deprec = p_run.get('tempo_depreciacao_equipamento_meses', 24)
        deprec = (p_run['capex_inicial'] / tempo_deprec) if mes < tempo_deprec else 0
        dados['depreciacao'][mes] = deprec

        dados['ebit'][mes] = dados['ebitda'][mes] - deprec
        dados['lucro_liquido'][mes] = dados['ebit'][mes]
        dados['margem_liquida'][mes] = (dados['lucro_liquido'][mes] / dados['receita_bruta'][mes]) * 100 if dados['receita_bruta'][mes] > 0 else 0

        # --------------------------
        # 3.10 FLUXO DE CAIXA
        # --------------------------
        if mes < p_run['meses_aporte']:
            dados['aportes_capital'][mes] = p_run['aporte_mensal']
        else:
            dados['aportes_capital'][mes] = 0.0

        fluxo_invest = 0.0
        # CAPEX sai do fluxo de caixa no mês 0 (não foi subtraído do caixa_inicial no Bloco 2).
        # Comentário ajustado: o saldo inicial foi mantido e o CAPEX será descontado aqui.
        if mes == 0:
            fluxo_invest -= p_run['capex_inicial']
        if mes == 24:
            fluxo_invest -= p_run.get('capex_recorrente_24m', 0)

        dados['capex'][mes] = fluxo_invest

        variacao_caixa = (dados['lucro_liquido'][mes] + deprec) + fluxo_invest + dados['aportes_capital'][mes]
        caixa_atual += variacao_caixa
        dados['caixa'][mes] = caixa_atual

        dados['burn_rate'][mes] = abs(dados['lucro_liquido'][mes]) if dados['lucro_liquido'][mes] < 0 else 0
        dados['runway_meses'][mes] = caixa_atual / dados['burn_rate'][mes] if dados['burn_rate'][mes] > 0 else 999

        # --------------------------
        # 3.11 UNIT ECONOMICS
        # --------------------------
        if dados['novos_pagantes_total'][mes] > 0:
            dados['cac_blended'][mes] = dados['gasto_marketing'][mes] / dados['novos_pagantes_total'][mes]
        else:
            dados['cac_blended'][mes] = 0.0

        if (usuarios_ativos > 0 and churn_rate > 0):
            ltv_val = (dados['margem_bruta'][mes] / usuarios_ativos) * (1 / churn_rate)
        else:
            ltv_val = 0.0

        dados['ltv'][mes] = ltv_val
        dados['ltv_cac'][mes] = ltv_val / dados['cac_blended'][mes] if dados['cac_blended'][mes] > 0 else 0.0

    # --------------------------------------------------------------------
    # BLOCO 4: OUTPUT
    # --------------------------------------------------------------------
    df_mensal = pd.DataFrame(dados)
    df_mensal.insert(0, 'mes', range(1, meses + 1))
    df_anual = df_mensal.groupby(df_mensal.index // 12).sum()

    metricas = {
        'usuarios_final': int(usuarios_ativos),
        'caixa_final': caixa_atual,
        'mrr_final': float(dados['mrr'][-1]),
        'arr_final': float(dados['arr'][-1]),
        'cac_medio': float(df_mensal['cac_blended'].replace(0, np.nan).mean()),
        'ltv_media': float(df_mensal['ltv'].mean()),
        'churn_medio': float(df_mensal['churn_rate'].mean() * 100),
        'margem_bruta_media': float(df_mensal['margem_bruta_pct'].mean()),
        'ebitda_margin_media': float(df_mensal['ebitda_margin'].mean())
    }

    return df_mensal, df_anual, metricas, alertas


In [5]:
# Executar o motor financeiro
df_mensal, df_anual, metricas, alertas, analise, insights = executar_motor_fintech_v9_2(PREMISSAS)

# Imprimir análise de viabilidade
print("="*80)
print("🩺 ANÁLISE DE VIABILIDADE")
print("="*80)
print(f"Status: {'✅ VIÁVEL' if analise['viavel'] else '❌ NÃO VIÁVEL'}")

if analise['alertas']:
    print("\n⚠️ ALERTAS:")
    for alerta in analise['alertas']:
        print(f"   • {alerta}")

if analise['recomendacoes']:
    print("\n💡 RECOMENDAÇÕES:")
    for rec in analise['recomendacoes']:
        print(f"   • {rec}")

if insights:
    print("\n🔍 INSIGHTS:")
    for insight in insights:
        print(f"   • {insight}")

print("\n📊 MÉTRICAS FINAIS:")
for key, value in metricas.items():
    if isinstance(value, float):
        print(f"   • {key}: {value:.2f}")
    else:
        print(f"   • {key}: {value}")

NameError: name 'executar_motor_fintech_v9_2' is not defined

In [ ]:
# CÉLULA 05: SETUP VISUAL, DESIGN SYSTEM & FORMATADORES (ESTRUTURA UX/UI)
# ============================================================================
# OBJETIVO: Garantir que todos os gráficos e tabelas tenham padrão institucional.
# ALCANCE: Define paletas de cores, templates do Plotly e funções de texto.
# STATUS: AUDITADO E APROVADO (GOLD)
# ============================================================================

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------------
# 1. DESIGN SYSTEM (PALETA SEMÂNTICA)
# ----------------------------------------------------------------------------
CORES = {
    # Estruturais
    'primary': '#0F172A',      # Navy Blue
    'secondary': '#64748B',    # Slate
    'background': '#FFFFFF',   # Branco
    'grid': '#F1F5F9',         # Cinza Claro
    
    # Financeiro (Semântica Contábil)
    'receita': '#10B981',      # Emerald 500 (Positivo)
    'despesa': '#EF4444',      # Red 500 (Negativo)
    'lucro': '#3B82F6',        # Blue 500
    'ebitda': '#6366F1',       # Indigo 500
    'caixa': '#8B5CF6',        # Violet 500
    'success': '#10B981',      # Verde (sucesso)
    'danger': '#EF4444',       # Vermelho (perigo)
    'info': '#3B82F6',         # Azul (info)
    'warning': '#F59E0B',      # Laranja (alerta)
    
    # Growth & Produto
    'usuario_ativo': '#0EA5E9', 
    'churn': '#F59E0B',         
    'trafego': '#94A3B8',       
    
    # Canais
    'org': '#10B981',           
    'ads': '#EC4899',           
    'afiliado': '#F97316',      
    'marketing_pago': '#3B82F6',
    'marketing_organico': '#10B981',
    
    # Tiers
    'lite': '#A7F3D0',          
    'trader': '#34D399',        
    'pro': '#059669',
    
    # Infraestrutura
    'infra': '#64748B'
}

# ----------------------------------------------------------------------------
# 2. HELPER: CONVERSÃO HEX PARA RGBA
# ----------------------------------------------------------------------------
def hex_to_rgba(hex_color, alpha=0.125):
    """
    Converte cor hexadecimal para rgba com transparência.
    Necessário para gráficos Sankey que não aceitam formato #RRGGBBAA
    """
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r}, {g}, {b}, {alpha})'

# ----------------------------------------------------------------------------
# 3. CONFIGURAÇÃO DE TEMPLATE PLOTLY
# ----------------------------------------------------------------------------
def configurar_tema_plotly():
    """
    Cria e registra o tema 'fintech_pro' no Plotly.
    """
    # Create Template object
    template = go.layout.Template()
    
    # Define Layout
    template.layout = go.Layout(
        font=dict(family="Inter, Roboto, Arial, sans-serif", size=12, color=CORES['primary']),
        title=dict(font=dict(size=18, color=CORES['primary'], family="Inter, sans-serif")),
        plot_bgcolor=CORES['background'],
        paper_bgcolor=CORES['background'],
        xaxis=dict(
            showgrid=False, zeroline=False, showline=True, 
            linecolor=CORES['grid'], tickfont=dict(color=CORES['secondary'])
        ),
        yaxis=dict(
            showgrid=True, gridcolor=CORES['grid'], zeroline=False, showline=False,
            tickfont=dict(color=CORES['secondary'])
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
        hoverlabel=dict(bgcolor=CORES['primary'], font=dict(color='white')),
        margin=dict(t=60, l=40, r=40, b=40),
        colorway=[
            CORES['receita'], CORES['despesa'], CORES['lucro'], 
            CORES['ebitda'], CORES['caixa'], CORES['usuario_ativo']
        ]
    )
    
    pio.templates["fintech_pro"] = template
    pio.templates.default = "fintech_pro"

configurar_tema_plotly()

# ----------------------------------------------------------------------------
# 4. HELPER: FORMATAÇÃO
# ----------------------------------------------------------------------------
def fmt_moeda(valor):
    if abs(valor) >= 1_000_000:
        return f"R$ {valor/1_000_000:.1f}M"
    elif abs(valor) >= 1_000:
        return f"R$ {valor/1_000:.1f}k"
    else:
        return f"R$ {valor:.0f}"


In [ ]:
# CÉLULA 05A: EXECUÇÃO BASE (CORRIGIDA)
# ============================================================================
# TEMPO ESTIMADO: ~30 segundos
# OUTPUT: df_mensal, df_anual, metricas_chave, alertas
# CORREÇÃO: Ajuste de chaves do dicionário (singular/plural) e cálculo de ROI
# ============================================================================

import time
import numpy as np
inicio = time.time()

print("="*80)
print("🚀 EXECUTANDO SIMULAÇÃO BASE (CENÁRIO DETERMINÍSTICO)")
print("="*80)
print(f"\n📅 Período: {PREMISSAS['data_inicio']} a {PREMISSAS['meses_projecao']} meses")
print(f"🎲 Seed: {PREMISSAS['seed_fixa']} (reproduzível)")
print(f"\n⏳ Processando...\n")

# ============================================================================
# EXECUÇÃO DO MOTOR
# ============================================================================
df_mensal, df_anual, metricas_chave, alertas = executar_motor_fintech_v7_investor_grade(
    PREMISSAS, 
    seed=PREMISSAS['seed_fixa'],
    modo_debug=False
)

# ============================================================================
# 🚑 PATCH DE SEGURANÇA (GARANTIA DE CHAVES)
# ============================================================================
# Garante que chaves críticas existam para evitar KeyError no print
if 'usuarios_final' not in metricas_chave:
    metricas_chave['usuarios_final'] = int(df_mensal['usuarios_ativos'].iloc[-1])

# Cálculo manual de Investimento e ROI se não vier do motor
if 'total_investido' not in metricas_chave:
    metricas_chave['total_investido'] = df_mensal['aportes_capital'].sum() + PREMISSAS['caixa_inicial']

if 'roi' not in metricas_chave:
    lucro_acumulado = df_mensal['lucro_liquido'].sum()
    investido = metricas_chave['total_investido']
    metricas_chave['roi'] = (lucro_acumulado / investido * 100) if investido > 0 else 0

if 'margem_bruta_media' not in metricas_chave:
    metricas_chave['margem_bruta_media'] = df_mensal['margem_bruta_pct'].mean()

if 'ebitda_margin_media' not in metricas_chave:
    metricas_chave['ebitda_margin_media'] = df_mensal['ebitda_margin'].mean()

tempo_exec = time.time() - inicio

# ============================================================================
# SUMÁRIO DE EXECUÇÃO
# ============================================================================
print("="*80)
print("✅ SIMULAÇÃO CONCLUÍDA!")
print("="*80)

print(f"\n⏱️  PERFORMANCE:")
print(f"   Tempo de Execução: {tempo_exec:.2f}s")
print(f"   Meses Simulados: {PREMISSAS['meses_projecao']}")
print(f"   Métricas Calculadas: {len(df_mensal.columns)}")

print(f"\n💰 RESULTADO FINANCEIRO (MÊS 36):")
print(f"   MRR Final: {formatar_moeda(metricas_chave['mrr_final'])}")
print(f"   ARR Final: {formatar_moeda(metricas_chave['arr_final'])}")
# CORRIGIDO AQUI: usuarios_final (singular)
print(f"   Usuários Finais: {int(metricas_chave['usuarios_final']):,}")
print(f"   Caixa Final: {formatar_moeda(metricas_chave['caixa_final'])}")

print(f"\n📊 UNIT ECONOMICS:")
print(f"   LTV/CAC Médio: {metricas_chave.get('ltv_cac_medio', df_mensal['ltv_cac'].mean()):.2f}x")
print(f"   Payback Médio: {metricas_chave.get('payback_medio_meses', df_mensal['payback_meses'].mean()):.1f} meses")
print(f"   CAC Médio: {formatar_moeda(metricas_chave['cac_medio'])}")
print(f"   Churn Médio: {metricas_chave['churn_medio']:.1f}%")

print(f"\n📈 MARGENS:")
print(f"   Margem Bruta: {metricas_chave['margem_bruta_media']:.1f}%")
print(f"   EBITDA Margin: {metricas_chave['ebitda_margin_media']:.1f}%")

print(f"\n💸 INVESTIMENTO & RETORNO:")
print(f"   Total Investido: {formatar_moeda(metricas_chave['total_investido'])}")
print(f"   ROI Estimado: {metricas_chave['roi']:.1f}%")

# ============================================================================
# ALERTAS & WARNINGS
# ============================================================================
if len(alertas) > 0:
    print(f"\n⚠️  ALERTAS IDENTIFICADOS ({len(alertas)}):")
    for i, alerta in enumerate(alertas[:5], 1):  # Top 5
        tipo_emoji = {
            'caixa_negativo': '💸',
            'ltv_cac_critico': '📉',
            'churn_alto': '⚠️',
            'burn_rate_alto': '🔥',
            'opex_zero': '❌'
        }.get(alerta['tipo'], '⚠️')
        
        print(f"   {tipo_emoji} Mês {alerta['mes']}: {alerta['mensagem']}")
    
    if len(alertas) > 5:
        print(f"   ... e mais {len(alertas) - 5} alertas")
else:
    print(f"\n✅ NENHUM ALERTA CRÍTICO IDENTIFICADO")

# ============================================================================
# STATUS DOS DATAFRAMES
# ============================================================================
print(f"\n📋 DATAFRAMES CRIADOS:")
print(f"   • df_mensal: {df_mensal.shape[0]} linhas × {df_mensal.shape[1]} colunas")
print(f"   • df_anual: {df_anual.shape[0]} linhas × {df_anual.shape[1]} colunas")
print(f"   • metricas_chave: {len(metricas_chave)} KPIs consolidados")

print(f"\n📊 PREVIEW df_mensal (Primeiras 3 + Últimas 3 linhas):")
print("\n" + "="*80)

# Colunas principais para preview
colunas_preview = [
    'mes', 'usuarios_ativos', 'mrr', 'caixa', 
    'cac_blended', 'ltv_cac', 'churn_rate'
]

# Verifica quais colunas existem
colunas_disponiveis = [col for col in colunas_preview if col in df_mensal.columns]

# Preview formatado
df_preview_head = df_mensal[colunas_disponiveis].head(3).copy()
df_preview_tail = df_mensal[colunas_disponiveis].tail(3).copy()

# Formata valores monetários
for col in ['mrr', 'caixa']:
    if col in df_preview_head.columns:
        df_preview_head[col] = df_preview_head[col].apply(lambda x: f"R$ {x:,.0f}")
        df_preview_tail[col] = df_preview_tail[col].apply(lambda x: f"R$ {x:,.0f}")

# Formata valores percentuais
for col in ['churn_rate']:
    if col in df_preview_head.columns:
        df_preview_head[col] = df_preview_head[col].apply(lambda x: f"{x*100:.1f}%")
        df_preview_tail[col] = df_preview_tail[col].apply(lambda x: f"{x*100:.1f}%")

# Formata inteiros
for col in ['usuarios_ativos']:
    if col in df_preview_head.columns:
        df_preview_head[col] = df_preview_head[col].apply(lambda x: f"{int(x):,}")
        df_preview_tail[col] = df_preview_tail[col].apply(lambda x: f"{int(x):,}")

# Formata decimais
for col in ['cac_blended', 'ltv_cac']:
    if col in df_preview_head.columns:
        df_preview_head[col] = df_preview_head[col].apply(lambda x: f"{x:.2f}")
        df_preview_tail[col] = df_preview_tail[col].apply(lambda x: f"{x:.2f}")

print(df_preview_head.to_string(index=False))
print("...")
print(df_preview_tail.to_string(index=False))

print("\n" + "="*80)

# ============================================================================
# ANÁLISE RÁPIDA DE VIABILIDADE
# ============================================================================
print(f"\n🎯 ANÁLISE DE VIABILIDADE:")

# Breakeven
meses_positivos = (df_mensal['lucro_liquido'] > 0).sum()
mes_breakeven = None
if meses_positivos > 0:
    idx_breakeven = (df_mensal['lucro_liquido'] > 0).idxmax()
    mes_breakeven = df_mensal.loc[idx_breakeven, 'mes']
    print(f"   ✅ Breakeven alcançado no mês {int(mes_breakeven)}")
else:
    print(f"   ⚠️  Breakeven não alcançado nos {PREMISSAS['meses_projecao']} meses")

# Runway
runway_final = metricas_chave.get('runway_final', df_mensal['runway_meses'].iloc[-1])
if runway_final > 12:
    print(f"   ✅ Runway saudável: {runway_final:.0f} meses")
elif runway_final > 6:
    print(f"   ⚠️  Runway moderado: {runway_final:.0f} meses")
else:
    print(f"   🔴 Runway crítico: {runway_final:.0f} meses")

# LTV/CAC
ltv_cac_final = df_mensal['ltv_cac'].iloc[-1]
if ltv_cac_final >= PREMISSAS['benchmark_ltv_cac_excelente']:
    print(f"   ✅ LTV/CAC excelente: {ltv_cac_final:.2f}x (≥ {PREMISSAS['benchmark_ltv_cac_excelente']}x)")
elif ltv_cac_final >= PREMISSAS['benchmark_ltv_cac_atencao']:
    print(f"   ✅ LTV/CAC saudável: {ltv_cac_final:.2f}x (≥ {PREMISSAS['benchmark_ltv_cac_atencao']}x)")
elif ltv_cac_final >= PREMISSAS['benchmark_ltv_cac_critico']:
    print(f"   ⚠️  LTV/CAC atenção: {ltv_cac_final:.2f}x (< {PREMISSAS['benchmark_ltv_cac_atencao']}x)")
else:
    print(f"   🔴 LTV/CAC crítico: {ltv_cac_final:.2f}x (< {PREMISSAS['benchmark_ltv_cac_critico']}x)")

# Churn
churn_final = df_mensal['churn_rate'].iloc[-1] * 100
if churn_final <= PREMISSAS['benchmark_churn_atencao_pct']:
    print(f"   ✅ Churn saudável: {churn_final:.1f}% (≤ {PREMISSAS['benchmark_churn_atencao_pct']}%)")
elif churn_final <= PREMISSAS['benchmark_churn_critico_pct']:
    print(f"   ⚠️  Churn atenção: {churn_final:.1f}% (≤ {PREMISSAS['benchmark_churn_critico_pct']}%)")
else:
    print(f"   🔴 Churn crítico: {churn_final:.1f}% (> {PREMISSAS['benchmark_churn_critico_pct']}%)")

# Margem Bruta
margem_bruta_final = df_mensal['margem_bruta_pct'].iloc[-1]
if margem_bruta_final >= PREMISSAS['benchmark_margem_alvo_pct']:
    print(f"   ✅ Margem bruta excelente: {margem_bruta_final:.1f}% (≥ {PREMISSAS['benchmark_margem_alvo_pct']}%)")
elif margem_bruta_final >= PREMISSAS['benchmark_margem_critica_pct']:
    print(f"   ⚠️  Margem bruta moderada: {margem_bruta_final:.1f}% (≥ {PREMISSAS['benchmark_margem_critica_pct']}%)")
else:
    print(f"   🔴 Margem bruta crítica: {margem_bruta_final:.1f}% (< {PREMISSAS['benchmark_margem_critica_pct']}%)")

# ============================================================================
# PRÓXIMO PASSO
# ============================================================================
print(f"\n" + "="*80)
print("📌 PRÓXIMO PASSO: Execute a CÉLULA 05B (Monte Carlo)")
print("="*80)

🚀 EXECUTANDO SIMULAÇÃO BASE (CENÁRIO DETERMINÍSTICO)

📅 Período: 2025-11-01 a 36 meses
🎲 Seed: 42 (reproduzível)

⏳ Processando...

✅ SIMULAÇÃO CONCLUÍDA!

⏱️  PERFORMANCE:
   Tempo de Execução: 0.06s
   Meses Simulados: 36
   Métricas Calculadas: 101

💰 RESULTADO FINANCEIRO (MÊS 36):
   MRR Final: R$ 94.651,00
   ARR Final: R$ 1.135.812,00
   Usuários Finais: 990
   Caixa Final: R$ 62.869,47

📊 UNIT ECONOMICS:
   LTV/CAC Médio: 3.78x
   Payback Médio: 0.0 meses
   CAC Médio: R$ 270,78
   Churn Médio: 9.4%

📈 MARGENS:
   Margem Bruta: 74.1%
   EBITDA Margin: -18.7%

💸 INVESTIMENTO & RETORNO:
   Total Investido: R$ 20.000,00
   ROI Estimado: 274.3%

✅ NENHUM ALERTA CRÍTICO IDENTIFICADO

📋 DATAFRAMES CRIADOS:
   • df_mensal: 36 linhas × 101 colunas
   • df_anual: 3 linhas × 101 colunas
   • metricas_chave: 11 KPIs consolidados

📊 PREVIEW df_mensal (Primeiras 3 + Últimas 3 linhas):

 mes usuarios_ativos      mrr     caixa cac_blended ltv_cac churn_rate
   1               8   R$ 719 R$ -8,

In [ ]:
# CÉLULA 05B: MONTE CARLO (SIMULAÇÃO ESTOCÁSTICA COM 5.000 CENÁRIOS)
# ============================================================================
# TEMPO ESTIMADO: ~5 a 10 minutos
# CORREÇÃO: Tratamento de erros em arrays vazios (LTV/CAC)
# ============================================================================

import pickle
import os
import time
import numpy as np
from tqdm import tqdm
from datetime import datetime

# ============================================================================
# VERIFICAÇÃO DE ARQUIVO EXISTENTE
# ============================================================================
arquivo_mc = 'mc_resultados.pkl'

# Para forçar reprocessamento com as novas premissas, deletamos o antigo se existir
if os.path.exists(arquivo_mc):
    try:
        # Verifica se o arquivo é compatível com a versão atual das premissas
        with open(arquivo_mc, 'rb') as f:
            dados_antigos = pickle.load(f)
        
        # Se for muito antigo ou premissas mudaram drasticamente, deleta
        # Aqui vamos forçar re-execução para garantir que use o cenário Bootstrapping
        print(f"♻️  Arquivo de Monte Carlo encontrado ({dados_antigos.get('timestamp')}), mas vamos recalcular para o cenário atual.")
        os.remove(arquivo_mc)
    except:
        pass

print("="*80)
print("🎲 EXECUTANDO MONTE CARLO (5.000 SIMULAÇÕES)")
print("="*80)
print(f"\n⚠️  ATENÇÃO: Este processo levará alguns minutos.")
print(f"   Você pode interromper com Ctrl+C. O progresso será salvo.\n")

# ========================================================================
# FUNÇÃO DE MONTE CARLO
# ========================================================================
def executar_monte_carlo_v6(premissas, n_simulacoes=5000):
    """
    Executa simulações estocásticas variando parâmetros críticos.
    Versão robusta contra NaNs e Zeros.
    """
    np.random.seed(premissas['seed_fixa'])
    
    # Arrays para armazenar resultados
    caixa_final_array = np.zeros(n_simulacoes)
    mrr_final_array = np.zeros(n_simulacoes)
    roi_array = np.zeros(n_simulacoes)
    ltv_cac_array = np.zeros(n_simulacoes)
    
    # Distribuições temporais
    caixa_mensal_dist = np.zeros((n_simulacoes, premissas['meses_projecao']))
    mrr_mensal_dist = np.zeros((n_simulacoes, premissas['meses_projecao']))
    
    print(f"🔧 Preparando {n_simulacoes:,} simulações...")
    
    # Progress bar
    with tqdm(total=n_simulacoes, desc="Simulando", unit="sim") as pbar:
        for sim in range(n_simulacoes):
            # Cria cópia das premissas
            p_var = premissas.copy()
            
            # --- VARIAÇÕES ESTOCÁSTICAS (PERTURBAÇÕES) ---
            
            # 1. Churn (Normal, std=20%)
            std_churn = premissas.get('mc_std_churn', 0.20)
            p_var['churn_inicial'] = max(0.01, min(0.30, 
                np.random.normal(premissas['churn_inicial'], premissas['churn_inicial'] * std_churn)
            ))
            
            # 2. Conversões (Normal, std=20%)
            std_conv = premissas.get('mc_std_trial_pag', 0.20)
            p_var['taxa_trial_para_pagante'] = max(0.01, min(0.50,
                np.random.normal(premissas['taxa_trial_para_pagante'], premissas['taxa_trial_para_pagante'] * std_conv)
            ))
            
            # 3. CPC Marketing (Normal, std=15%)
            std_mkt = premissas.get('mc_std_marketing', 0.15)
            # Aplica um fator global de custo de marketing para a simulação
            fator_mkt = max(0.5, np.random.normal(1.0, std_mkt))
            for canal in ['instagram', 'facebook', 'youtube', 'google']:
                key = f'cpc_{canal}'
                if key in p_var:
                    p_var[key] = p_var[key] * fator_mkt

            # 4. Crescimento (Varia a taxa semanal)
            # Removemos a lógica de "taxa separada" e variamos a taxa base definida
            std_cresc = premissas.get('mc_std_cresc_traf', 0.25)
            taxa_base = premissas.get('taxa_crescimento_semanal_m0_m6', 0.035)
            p_var['taxa_crescimento_semanal_m0_m6'] = max(0.005, 
                np.random.normal(taxa_base, taxa_base * std_cresc)
            )

            # --- EXECUÇÃO ---
            try:
                # Chama o motor (usando a versão V7/V6 disponível)
                # Importante: Modo debug desligado para performance
                df_sim, _, metricas_sim, _ = executar_motor_fintech_v7_investor_grade(
                    p_var, 
                    seed=premissas['seed_fixa'] + sim, 
                    modo_debug=False
                )
                
                # Coleta resultados
                caixa_final_array[sim] = metricas_sim.get('caixa_final', 0)
                mrr_final_array[sim] = metricas_sim.get('mrr_final', 0)
                
                # ROI Calculado na hora para garantir
                investido = premissas['caixa_inicial'] + df_sim['aportes_capital'].sum()
                lucro = df_sim['lucro_liquido'].sum()
                roi_array[sim] = (lucro / investido * 100) if investido > 0 else 0
                
                ltv_cac_array[sim] = metricas_sim.get('ltv_cac_medio', 0)
                
                # Séries temporais
                if len(df_sim) >= premissas['meses_projecao']:
                    caixa_mensal_dist[sim, :] = df_sim['caixa'].values[:premissas['meses_projecao']]
                    mrr_mensal_dist[sim, :] = df_sim['mrr'].values[:premissas['meses_projecao']]
                
            except Exception as e:
                # Fallback seguro
                caixa_final_array[sim] = np.nan
            
            pbar.update(1)
    
    # --- CÁLCULO DE ESTATÍSTICAS (PERCENTIS) ---
    
    # Helper para percentil seguro (ignora NaNs)
    def safe_percentile(arr, q):
        arr_valid = arr[~np.isnan(arr)]
        if len(arr_valid) == 0: return 0.0
        return np.percentile(arr_valid, q)

    # LTV/CAC (Filtra zeros e NaNs para não distorcer)
    ltv_cac_validos = ltv_cac_array[(~np.isnan(ltv_cac_array)) & (ltv_cac_array > 0)]
    if len(ltv_cac_validos) == 0:
        ltv_p10, ltv_p50, ltv_p90 = 0, 0, 0
    else:
        ltv_p10 = np.percentile(ltv_cac_validos, 10)
        ltv_p50 = np.percentile(ltv_cac_validos, 50)
        ltv_p90 = np.percentile(ltv_cac_validos, 90)

    resultados = {
        'n_simulacoes': n_simulacoes,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        
        # Caixa
        'caixa_p10': safe_percentile(caixa_final_array, 10),
        'caixa_p50': safe_percentile(caixa_final_array, 50),
        'caixa_p90': safe_percentile(caixa_final_array, 90),
        'caixa_array': caixa_final_array,
        
        # MRR
        'mrr_p10': safe_percentile(mrr_final_array, 10),
        'mrr_p50': safe_percentile(mrr_final_array, 50),
        'mrr_p90': safe_percentile(mrr_final_array, 90),
        
        # ROI
        'roi_p10': safe_percentile(roi_array, 10),
        'roi_p50': safe_percentile(roi_array, 50),
        'roi_p90': safe_percentile(roi_array, 90),
        
        # LTV/CAC
        'ltv_cac_p10': ltv_p10,
        'ltv_cac_p50': ltv_p50,
        'ltv_cac_p90': ltv_p90,
        
        # Séries (Fan Chart) - Axis 0 = ao longo das simulações
        'caixa_mensal_p10': np.nanpercentile(caixa_mensal_dist, 10, axis=0),
        'caixa_mensal_p50': np.nanpercentile(caixa_mensal_dist, 50, axis=0),
        'caixa_mensal_p90': np.nanpercentile(caixa_mensal_dist, 90, axis=0),
        
        # Probabilidades
        'prob_caixa_negativo': np.mean(caixa_final_array < 0) * 100,
        'prob_roi_positivo': np.mean(roi_array > 0) * 100
    }
    
    return resultados

# ========================================================================
# EXECUÇÃO E SALVAMENTO
# ========================================================================
inicio_mc = time.time()

mc_resultados = executar_monte_carlo_v6(
    PREMISSAS, 
    n_simulacoes=PREMISSAS['mc_n_simulacoes']
)

# Salvar
with open(arquivo_mc, 'wb') as f:
    pickle.dump(mc_resultados, f)

# ========================================================================
# REPORT
# ========================================================================
print(f"\n" + "="*80)
print("📊 RESULTADOS DO MONTE CARLO (BOOTSTRAPPING RISK)")
print("="*80)

print(f"\n💰 CAIXA FINAL (Mês 36):")
print(f"   P10 (Pessimista): {formatar_moeda(mc_resultados['caixa_p10'])}")
print(f"   P50 (Mediana):    {formatar_moeda(mc_resultados['caixa_p50'])}")
print(f"   P90 (Otimista):   {formatar_moeda(mc_resultados['caixa_p90'])}")

print(f"\n💸 ROI (%):")
print(f"   P10: {mc_resultados['roi_p10']:.1f}%")
print(f"   P50: {mc_resultados['roi_p50']:.1f}%")
print(f"   P90: {mc_resultados['roi_p90']:.1f}%")

print(f"\n⚠️  ANÁLISE DE RISCO:")
print(f"   Probabilidade de Caixa Negativo: {mc_resultados['prob_caixa_negativo']:.1f}%")
print(f"   Probabilidade de Lucro (ROI > 0): {mc_resultados['prob_roi_positivo']:.1f}%")

print(f"\n" + "="*80)
print("✅ MONTE CARLO CONCLUÍDO!")

🎲 EXECUTANDO MONTE CARLO (5.000 SIMULAÇÕES)

⚠️  ATENÇÃO: Este processo levará alguns minutos.
   Você pode interromper com Ctrl+C. O progresso será salvo.

🔧 Preparando 100 simulações...


Simulando: 100%|██████████| 100/100 [00:05<00:00, 16.96sim/s]


📊 RESULTADOS DO MONTE CARLO (BOOTSTRAPPING RISK)

💰 CAIXA FINAL (Mês 36):
   P10 (Pessimista): R$ 32.476,43
   P50 (Mediana):    R$ 82.919,72
   P90 (Otimista):   R$ 412.648,14

💸 ROI (%):
   P10: 122.4%
   P50: 374.6%
   P90: 2023.2%

⚠️  ANÁLISE DE RISCO:
   Probabilidade de Caixa Negativo: 0.0%
   Probabilidade de Lucro (ROI > 0): 98.0%

✅ MONTE CARLO CONCLUÍDO!


In [ ]:
#CÉLULA 05C - CALENDÁRIO DE METAS SEMANAL (SEMANAS 1-26)
"""
Para IMPRIMIR E COLAR NA PAREDE do escritório

NÃO é análise, NÃO é comparação real vs projetado, NÃO é simulação.
É SOMENTE: "Essa semana preciso atingir X usuários, Y MRR, Z CAC"

Baseado em benchmarks REALISTAS de fintech B2C trading
Fonte: docs/benchmarks_da_industria.md
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===========================================================================
# METAS SEMANAIS BASEADAS EM BENCHMARKS REALISTAS
# ===========================================================================

def gerar_metas_semanais_realistas(data_inicio='2025-11-01'):
    """
    Gera metas semanais ALCANÇÁVEIS para primeiros 6 meses (26 semanas)
    
    BENCHMARKS USADOS (fonte: benchmarks_da_industria.md):
    - Churn fintech traders: 6-12% mensal (usamos 10% → 2.5% semanal)
    - CAC inicial fintech: R$250-750 (usamos R$500 S1 → R$200 S26)  
    - LTV/CAC fase validação: 2-4x (usamos progressivo 2x → 4x)
    - Crescimento conservador: 15% M-o-M = ~3.5% semanal
    """
    
    # Configuração inicial
    data_inicial = datetime.strptime(data_inicio, '%Y-%m-%d')
    
    # Metas semanais (26 semanas = 6 meses)
    semanas = []
    
    # Premissas base (REALISTAS)
    usuarios_s1 = 20  # Começar com 20 usuários pagantes
    arpu = 97  # ARPU médio R$97 (conforme docs)
    
    # Taxa de crescimento semanal (baseado em 15% M-o-M = ~3.5% semanal)
    taxa_crescimento_semanal = 0.035  # 3.5% por semana
    
    for semana in range(1, 27):  # Semanas 1-26
        # Calcular data
        data_inicio_semana = data_inicial + timedelta(weeks=semana-1)
        data_fim_semana = data_inicio_semana + timedelta(days=6)
        
        # Usuários crescem 3.5% por semana
        usuarios_meta = int(usuarios_s1 * ((1 + taxa_crescimento_semanal) ** (semana - 1)))
        
        # MRR = usuários × ARPU
        mrr_meta = usuarios_meta * arpu
        
        # CAC diminui progressivamente (R$500 → R$200 em 26 semanas)
        # Lógica: otimização de campanhas leva tempo
        cac_maximo = 500 - ((semana - 1) * 12)  # -R$12 por semana
        
        # Churn semanal: 10% mensal = ~2.5% semanalначально, diminui para 1.8%
        churn_maximo_semanal = 2.5 - ((semana - 1) * 0.027)  # 2.5% → 1.8%
        
        # LTV progressivo (melhora com menor churn e maior retenção)
        # LTV = ARPU × Margem Bruta (85%) / Churn mensal
        churn_mensal_equivalente = churn_maximo_semanal * 4.33  # ~4.33 semanas/mês
        ltv_minimo = (arpu * 0.85) / (churn_mensal_equivalente / 100) if churn_mensal_equivalente > 0 else 1000
        
        # LTV/CAC ratio
        ltv_cac_minimo = ltv_minimo / cac_maximo if cac_maximo > 0 else 0
        
        # Conversão trial → pago (melhorar de 8% para 12% ao longo de 26 semanas)
        conversao_minima = 8 + ((semana - 1) * 0.15)  # 8% → 11.8%
        
        # Margem contribuição (melhorar de -50% para +15% em S26)
        # Lógica: custos fixos são diluídos com mais usuários
        margem_contrib_minima = -50 + ((semana - 1) * 2.5)  # -50% → +15%
        
        # Burn rate (queima de caixa semanal)
        # Diminui conforme receita cresce
        burn_rate_maximo = 3000 - ((semana - 1) * 80)  # R$3k → R$1k semanal
        
        # Runway (meses de sobrevivência)
        runway_minimo = 12 + (semana * 0.1)  # Manter sempre >12 meses
        
        semanas.append({
            'semana': semana,
            'data_inicio': data_inicio_semana.strftime('%d/%m'),
            'data_fim': data_fim_semana.strftime('%d/%m'),
            'usuarios_pagantes': usuarios_meta,
            'mrr': mrr_meta,
            'cac_maximo': int(cac_maximo),
            'churn_maximo_%': round(churn_maximo_semanal, 1),
            'ltv_minimo': int(ltv_minimo),
            'ltv_cac_minimo': round(ltv_cac_minimo, 1),
            'conversao_minima_%': round(conversao_minima, 1),
            'margem_contrib_minima_%': round(margem_contrib_minima, 1),
            'burn_rate_maximo': int(max(burn_rate_maximo, 0)),
            'runway_minimo_meses': round(runway_minimo, 1)
        })
    
    return pd.DataFrame(semanas)

# ===========================================================================
# VISUALIZAÇÃO 1: CALENDÁRIO IMPRIMÍVEL
# ===========================================================================

def calendario_semanal_imprimivel(df_metas):
    """
    Cria visualização bonita para IMPRIMIR e colar na parede
    """
    
    # CSS para impressão
    display(HTML("""
    <style>
    @media print {
        .pagebreak { page-break-before: always; }
    }
    .calendario-metas {
        font-family: 'Inter', 'Segoe UI', sans-serif;
        background: white;
        border: none;
        margin: 20px 0;
    }
    .semana-card {
        border-left: 4px solid #6366F1;
        padding: 15px 20px;
        margin: 15px 0;
        background: #F8FAFC;
        page-break-inside: avoid;
    }
    .semana-header {
        font-size: 18px;
        font-weight: 700;
        color: #0F172A;
        margin-bottom: 10px;
    }
    .meta-item {
        display: inline-block;
        width: 48%;
        padding: 8px 0;
        font-size: 14px;
    }
    .meta-label {
        color: #64748B;
        font-weight: 500;
    }
    .meta-valor {
        color: #0F172A;
        font-weight: 700;
        font-size: 16px;
    }
    .checkpoint {
        background: #FEF3C7;
        border-left-color: #F59E0B;
    }
    .titulo-calendario {
        text-align: center;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 30px;
        margin-bottom: 30px;
    }
    </style>
    """))
    
    # Título
    display(HTML("""
    <div class="titulo-calendario">
        <h1 style="margin: 0; font-size: 32px;">📅 CALENDÁRIO DE METAS</h1>
        <h2 style="margin: 10px 0 0 0; font-weight: 300; font-size: 20px;">
            Semanas 1-26 (6 Primeiros Meses)
        </h2>
        <p style="margin: 15px 0 0 0; font-size: 14px; opacity: 0.9;">
            Metas alcançáveis baseadas em benchmarks fintech B2C trading
        </p>
    </div>
    """))
    
    # Gerar cards para cada semana
    html_content = '<div class="calendario-metas">'
    
    for idx, row in df_metas.iterrows():
        # Checkpoints especiais
        is_checkpoint = row['semana'] in [4, 13, 26]  # S4 (M1), S13 (M3), S26 (M6)
        card_class = "semana-card checkpoint" if is_checkpoint else "semana-card"
        
        checkpoint_label = ""
        if row['semana'] == 4:
            checkpoint_label = " 🎯 CHECKPOINT MÊS 1"
        elif row['semana'] == 13:
            checkpoint_label = " 🎯 CHECKPOINT MÊS 3"
        elif row['semana'] == 26:
            checkpoint_label = " 🎯 DECISÃO MÊS 6"  
        
        html_content += f'''
        <div class="{card_class}">
            <div class="semana-header">
                Semana {row['semana']} ({row['data_inicio']} - {row['data_fim']}){checkpoint_label}
            </div>
            <div class="meta-item">
                <span class="meta-label">Usuários Pagantes:</span>
                <span class="meta-valor">{row['usuarios_pagantes']}</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">MRR (Receita Mensal):</span>
                <span class="meta-valor">R$ {row['mrr']/1000:.1f}k</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">CAC Máximo (Custo Aquisição):</span>
                <span class="meta-valor">R$ {row['cac_maximo']}</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">Churn Máximo (Cancelamento):</span>
                <span class="meta-valor">{row['churn_maximo_%']}%/semana</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">LTV Mínimo (Valor Cliente):</span>
                <span class="meta-valor">R$ {row['ltv_minimo']}</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">LTV/CAC Mínimo:</span>
                <span class="meta-valor">{row['ltv_cac_minimo']:.1f}x</span>
            </div>
        </div>
        '''
        
        # Page break a cada 8 semanas para impressão
        if row['semana'] in [8, 16, 24]:
            html_content += '<div class="pagebreak"></div>'
    
    html_content += '</div>'
    display(HTML(html_content))

# ===========================================================================
# VISUALIZAÇÃO 2: GRÁFICOS SEMANAIS
# ===========================================================================

def graficos_metas_semanais(df_metas):
    """
    Gráficos com granularidade SEMANAL (não mensal!)
    """
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Usuários Pagantes (Meta Semanal)",
            "MRR - Receita Recorrente (Meta Semanal)",
            "CAC Máximo vs LTV Mínimo (Semanal)",
            "Churn Máximo Permitido (Semanal)"
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": True}, {"secondary_y": False}]]
    )
    
    # Gráfico 1: Usuários
    fig.add_trace(
        go.Scatter(
            x=df_metas['semana'],
            y=df_metas['usuarios_pagantes'],
            name='Meta Usuários',
            line=dict(color='#6366F1', width=3),
            mode='lines+markers',
            marker=dict(size=6)
        ),
        row=1, col=1
    )
    
    # Gráfico 2: MRR
    fig.add_trace(
        go.Scatter(
            x=df_metas['semana'],
            y=df_metas['mrr'],
            name='Meta MRR',
            line=dict(color='#10B981', width=3),
            mode='lines+markers',
            marker=dict(size=6)
        ),
        row=1, col=2
    )
    
    # Gráfico 3: CAC vs LTV
    fig.add_trace(
        go.Scatter(
            x=df_metas['semana'],
            y=df_metas['cac_maximo'],
            name='CAC Máximo',
            line=dict(color='#EF4444', width=2),
            mode='lines'
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_metas['semana'],
            y=df_metas['ltv_minimo'],
            name='LTV Mínimo',
            line=dict(color='#10B981', width=2),
            mode='lines'
        ),
        row=2, col=1,
        secondary_y=True
    )
    
    # Gráfico 4: Churn
    fig.add_trace(
        go.Scatter(
            x=df_metas['semana'],
            y=df_metas['churn_maximo_%'],
            name='Churn Máximo',
            line=dict(color='#F59E0B', width=3),
            mode='lines+markers',
            marker=dict(size=6),
            fill='tozeroy',
            fillcolor='rgba(245, 158, 11, 0.1)'
        ),
        row=2, col=2
    )
    
    # Checkpoints verticais
    for checkpoint, nome in [(4, 'M1'), (13, 'M3'), (26, 'M6')]:
        for r, c in [(1,1), (1,2), (2,1), (2,2)]:
            fig.add_vline(
                x=checkpoint,
                line_dash='dot',
                line_color='#F59E0B',
                opacity=0.5,
                row=r, col=c
            )
    
    # Layout
    fig.update_xaxes(title_text="Semana", row=1, col=1)
    fig.update_xaxes(title_text="Semana", row=1, col=2)
    fig.update_xaxes(title_text="Semana", row=2, col=1)
    fig.update_xaxes(title_text="Semana", row=2, col=2)
    
    fig.update_yaxes(title_text="Usuários", row=1, col=1)
    fig.update_yaxes(title_text="MRR (R$)", row=1, col=2)
    fig.update_yaxes(title_text="CAC (R$)", row=2, col=1)
    fig.update_yaxes(title_text="LTV (R$)", row=2, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Churn (%)", row=2, col=2)
    
    fig.update_layout(
        title_text="<b>Metas Semanais (S1-S26) - Granularidade Semanal</b>",
        template='plotly_white',
        height=800,
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.show()

# ===========================================================================
# TABELA COMPLETA METAS SEMANAIS
# ===========================================================================

def tabela_completa_metas(df_metas):
    """
    Tabela Excel-style com TODAS as metas semanais
    """
    
    display(Markdown("""
    ## 📊 Tabela Completa - Todas as Metas Semanais
    
    **Para planilha/Excel**: Copiar e colar os dados abaixo
    
    **Legenda**:
    - **Usuários Pagantes**: Meta de assinantes ativos
    - **MRR**: Receita Recorrente Mensal (em R$)
    - **CAC Máximo**: Custo máximo para adquirir 1 cliente
    - **Churn Máximo**: Taxa máxima de cancelamento semanal
    - **LTV Mínimo**: Valor mínimo vitalício de 1 cliente
    - **LTV/CAC Mínimo**: Quantas vezes cliente vale vs custo
    """))
    
    # Formatar tabela
    df_display = df_metas[['semana', 'data_inicio', 'data_fim', 'usuarios_pagantes', 
                             'mrr', 'cac_maximo', 'churn_maximo_%', 'ltv_minimo', 'ltv_cac_minimo']].copy()
    
    df_display.columns = ['Semana', 'Início', 'Fim', 'Usuários', 'MRR (R$)', 
                           'CAC Máx', 'Churn Máx %', 'LTV Mín', 'LTV/CAC Mín']
    
    display(df_display)
    
    # Estatísticas resumo
    display(Markdown(f"""
    ### 📈 Resumo 6 Meses (26 Semanas)
    
    | Métrica | Semana 1 | Semana 26 | Variação |
    |---------|----------|-----------|----------|
    | Usuários | {df_metas.iloc[0]['usuarios_pagantes']} | {df_metas.iloc[-1]['usuarios_pagantes']} | +{((df_metas.iloc[-1]['usuarios_pagantes'] / df_metas.iloc[0]['usuarios_pagantes']) - 1) * 100:.0f}% |
    | MRR | R$ {df_metas.iloc[0]['mrr']/1000:.1f}k | R$ {df_metas.iloc[-1]['mrr']/1000:.1f}k | +{((df_metas.iloc[-1]['mrr'] / df_metas.iloc[0]['mrr']) - 1) * 100:.0f}% |
    | CAC | R$ {df_metas.iloc[0]['cac_maximo']} | R$ {df_metas.iloc[-1]['cac_maximo']} | -{((df_metas.iloc[0]['cac_maximo'] - df_metas.iloc[-1]['cac_maximo']) / df_metas.iloc[0]['cac_maximo']) * 100:.0f}% |
    | Churn | {df_metas.iloc[0]['churn_maximo_%']}% | {df_metas.iloc[-1]['churn_maximo_%']}% | -{df_metas.iloc[0]['churn_maximo_%'] - df_metas.iloc[-1]['churn_maximo_%']:.1f}pp |
    | LTV/CAC | {df_metas.iloc[0]['ltv_cac_minimo']:.1f}x | {df_metas.iloc[-1]['ltv_cac_minimo']:.1f}x | +{df_metas.iloc[-1]['ltv_cac_minimo'] - df_metas.iloc[0]['ltv_cac_minimo']:.1f}x |
    """))

# ===========================================================================
# EXECUTAR CALENDÁRIO COMPLETO
# ===========================================================================

def executar_calendario_metas_semanal(data_inicio='2025-11-01'):
    """
    Executa TUDO: gera metas + visualizações + tabelas
    """
    
    print("="*80)
    print("📅 GERANDO CALENDÁRIO DE METAS SEMANAIS (S1-S26)")
    print("="*80)
    print("")
    print("Baseado em:")
    print("  - Benchmarks fintech B2C trading")
    print("  - Churn realista: 10% mensal (não 5%)")
    print("  - CAC realista: R$500 → R$200 (não R$150 → R$50)")
    print("  - LTV/CAC: 2x → 4x (não 5x+)")
    print("")
    
    # Gerar metas
    df_metas = gerar_metas_semanais_realistas(data_inicio)
    
    # 1. Calendário imprimível
    calendario_semanal_imprimivel(df_metas)
    
    # 2. Gráficos semanais
    graficos_metas_semanais(df_metas)
    
    # 3. Tabela completa
    tabela_completa_metas(df_metas)
    
    print("")
    print("="*80)
    print("✅ CALENDÁRIO COMPLETO GERADO")
    print("="*80)
    print("")
    print("💡 Como usar:")
    print("  1. Imprimir calendário (página 1)")
    print("  2. Colar na parede do escritório")
    print("  3. Toda segunda-feira: ver meta da semana")
    print("  4. Toda sexta: conferir se atingiu")
    print("")
    
    return df_metas

# ===========================================================================
# EXEMPLO DE USO
# ===========================================================================

# Executar (descomentar linha abaixo)
df_metas_semanais = executar_calendario_metas_semanal()


📅 GERANDO CALENDÁRIO DE METAS SEMANAIS (S1-S26)

Baseado em:
  - Benchmarks fintech B2C trading
  - Churn realista: 10% mensal (não 5%)
  - CAC realista: R$500 → R$200 (não R$150 → R$50)
  - LTV/CAC: 2x → 4x (não 5x+)




    ## 📊 Tabela Completa - Todas as Metas Semanais

    **Para planilha/Excel**: Copiar e colar os dados abaixo

    **Legenda**:
    - **Usuários Pagantes**: Meta de assinantes ativos
    - **MRR**: Receita Recorrente Mensal (em R$)
    - **CAC Máximo**: Custo máximo para adquirir 1 cliente
    - **Churn Máximo**: Taxa máxima de cancelamento semanal
    - **LTV Mínimo**: Valor mínimo vitalício de 1 cliente
    - **LTV/CAC Mínimo**: Quantas vezes cliente vale vs custo
    

,Semana,Início,Fim,Usuários,MRR (R$),CAC Máx,Churn Máx %,LTV Mín,LTV/CAC Mín
0,1,01/11,07/11,20,1940,500,2.50,761,1.50
1,2,08/11,14/11,20,1940,488,2.50,769,1.60
2,3,15/11,21/11,21,2037,476,2.40,778,1.60
3,4,22/11,28/11,22,2134,464,2.40,787,1.70
4,5,29/11,05/12,22,2134,452,2.40,796,1.80
5,6,06/12,12/12,23,2231,440,2.40,805,1.80
6,7,13/12,19/12,24,2328,428,2.30,814,1.90
7,8,20/12,26/12,25,2425,416,2.30,823,2.00
8,9,27/12,02/01,26,2522,404,2.30,833,2.10
9,10,03/01,09/01,27,2619,392,2.30,843,2.20



    ### 📈 Resumo 6 Meses (26 Semanas)

    | Métrica | Semana 1 | Semana 26 | Variação |
    |---------|----------|-----------|----------|
    | Usuários | 20 | 47 | +135% |
    | MRR | R$ 1.9k | R$ 4.6k | +135% |
    | CAC | R$ 500 | R$ 200 | -60% |
    | Churn | 2.5% | 1.8% | -0.7pp |
    | LTV/CAC | 1.5x | 5.2x | +3.7x |
    


✅ CALENDÁRIO COMPLETO GERADO

💡 Como usar:
  1. Imprimir calendário (página 1)
  2. Colar na parede do escritório
  3. Toda segunda-feira: ver meta da semana
  4. Toda sexta: conferir se atingiu



In [ ]:
# CÉLULA 06 - EXECUTIVE SUMMARY V7.3 (CLEAN TABLES & CONTEXT)
# ============================================================================
# STATUS: AJUSTADO SOB DEMANDA
# 1. TABELAS: CSS removido completamente. Formatação pura.
# 2. GRÁFICOS: Adicionados ganchos de texto (Annotations) sobre o desenho original.
# 3. LÓGICA: Mantida original da V7.2.
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML

# ----------------------------------------------------------------------------
# 1. AUDITORIA MATEMÁTICA E CÁLCULO DE KPIs
# ----------------------------------------------------------------------------
print("="*80)
print("🧮 AUDITORIA DE CÁLCULOS (ROI & INVESTIMENTO)")
print("="*80)

total_aportes = df_mensal['aportes_capital'].sum()
caixa_inicial_real = PREMISSAS['caixa_inicial']
investimento_total_real = total_aportes + caixa_inicial_real
lucro_liquido_acumulado = df_mensal['lucro_liquido'].sum()
caixa_final = df_mensal['caixa'].iloc[-1]
roi_absoluto = lucro_liquido_acumulado

try:
    roi_pct = (roi_absoluto / investimento_total_real) * 100
except ZeroDivisionError:
    roi_pct = 0

print(f"\n🔍 DETALHAMENTO DO CÁLCULO DE ROI:")
print(f"   (+) Caixa Inicial:          R$ {caixa_inicial_real:,.2f}")
print(f"   (+) Total Aportes (36m):    R$ {total_aportes:,.2f}")
print(f"   (=) TOTAL INVESTIDO:        R$ {investimento_total_real:,.2f}")
print(f"   --------------------------------------------------")
print(f"   (+) Lucro Líq. Acumulado:   R$ {lucro_liquido_acumulado:,.2f}")
print(f"   (=) RETORNO TOTAL:          R$ {roi_absoluto:,.2f}")
print(f"   --------------------------------------------------")
print(f"   🎯 ROI FINAL VALIDADO:      {roi_pct:.1f}%")

# ----------------------------------------------------------------------------
# 2. HELPERS DE VISUALIZAÇÃO (TABELAS LIMPAS - SEM CSS)
# ----------------------------------------------------------------------------
def exibir_tabela_grafico(df_dados, colunas_map, titulo="Dados do Gráfico"):
    """
    Gera tabela HTML PURA (sem CSS de cores).
    Apenas formatação de valores.
    """
    df_temp = df_dados[list(colunas_map.keys())].copy()
    df_temp.rename(columns=colunas_map, inplace=True)
    
    # Formatação inteligente de valores
    for col in df_temp.columns:
        if any(x in col for x in ['R$', 'MRR', 'Caixa', 'Lucro', 'Aporte', 'Burn']):
            df_temp[col] = df_temp[col].apply(lambda x: f"R$ {x:,.2f}")
        elif any(x in col for x in ['%', 'Taxa', 'Margem']):
            df_temp[col] = df_temp[col].apply(lambda x: f"{x:.1f}%")
        elif any(x in col for x in ['Usuários', 'Meta', 'Real', 'Gap']):
            df_temp[col] = df_temp[col].apply(lambda x: f"{int(x):,}")
        elif 'Runway' in col:
            df_temp[col] = df_temp[col].apply(lambda x: f"{x:.1f}")
            
    # HTML Limpo (Standard)
    html = f"<h4>📋 {titulo}</h4>"
    # border=1 para garantir linhas visíveis na impressão, sem frescura
    html += df_temp.to_html(index=False, border=1, justify='right')
    display(HTML(html))

# ----------------------------------------------------------------------------
# 3. GRÁFICOS RESTAURADOS COM CONTEXTO ADICIONADO
# ----------------------------------------------------------------------------
display(Markdown("# 📊 DASHBOARD EXECUTIVO (V7.3 - OTIMIZADO)"))

# --- GRÁFICO 1: TRAÇÃO SEMANAL (S1-S26) ---
semanas_x = list(range(1, 27))
usuarios_m0 = PREMISSAS['usuarios_pagos_iniciais']
usuarios_m6 = df_mensal['usuarios_ativos'].iloc[5]
usuarios_semanais = [int(usuarios_m0 + (usuarios_m6 - usuarios_m0) * (s/26)) for s in semanas_x]
meta_semanal = [int(usuarios_m0 * (1.035 ** (s-1))) for s in semanas_x]

fig_tracao = go.Figure()

# Linhas Originais
fig_tracao.add_trace(go.Scatter(x=semanas_x, y=meta_semanal, name='Meta (3.5%/sem)', line=dict(color=PREMISSAS['cores']['ebitda'], width=2, dash='dash')))
fig_tracao.add_trace(go.Scatter(x=semanas_x, y=usuarios_semanais, name='Projeção Real', line=dict(color=PREMISSAS['cores']['primary'], width=4), mode='lines+markers', marker=dict(size=6)))

# Marcos Verticais Originais
fig_tracao.add_vline(x=4, line_dash="dot", line_color='gray')
fig_tracao.add_vline(x=13, line_dash="dot", line_color='gray')
fig_tracao.add_vline(x=26, line_dash="dot", line_color='gray')

# --- ADIÇÃO DE CONTEXTO (GANCHOS DE TEXTO) ---
# Mês 1
fig_tracao.add_annotation(
    x=4, y=usuarios_semanais[3],
    text=f"M1: {usuarios_semanais[3]}u",
    showarrow=True, arrowhead=1, ax=0, ay=-30, bgcolor="white"
)
# Mês 3
fig_tracao.add_annotation(
    x=13, y=usuarios_semanais[12],
    text=f"M3: {usuarios_semanais[12]}u",
    showarrow=True, arrowhead=1, ax=0, ay=-30, bgcolor="white"
)
# Final (M6)
gap = usuarios_semanais[-1] - meta_semanal[-1]
status = "BATIDO" if gap >= 0 else "FALHA"
fig_tracao.add_annotation(
    x=26, y=usuarios_semanais[-1],
    text=f"FINAL (S26): {usuarios_semanais[-1]}u<br>Gap: {gap} ({status})",
    showarrow=True, arrowhead=1, ax=-50, ay=0, bgcolor="white", bordercolor="black"
)

fig_tracao.update_layout(
    title="<b>🚀 Tração Inicial: Usuários Ativos (Semanal S1-S26)</b>",
    xaxis_title="Semana", yaxis_title="Usuários Ativos",
    template="plotly_white", height=450, hovermode="x unified"
)
fig_tracao.show()

# TABELA 1 LIMPA
df_semanal_view = pd.DataFrame({
    'Semana': semanas_x, 'Usuários Projeção': usuarios_semanais,
    'Meta': meta_semanal, 'Diferença': np.array(usuarios_semanais) - np.array(meta_semanal)
})
exibir_tabela_grafico(
    df_semanal_view[df_semanal_view['Semana'].isin([1, 4, 8, 13, 17, 21, 26])],
    {'Semana': 'Semana', 'Usuários Projeção': 'Usuários (Proj)', 'Meta': 'Meta', 'Diferença': 'Gap'},
    "Dados de Tração (Marcos Principais)"
)

# --- GRÁFICO 2: FINANCEIRO ---
fig_fin = go.Figure()

fig_fin.add_trace(go.Scatter(x=df_mensal['mes'], y=df_mensal['receita_bruta'], name='Receita Bruta', fill='tozeroy', line=dict(color=PREMISSAS['cores']['receita'], width=0), fillcolor="rgba(16, 185, 129, 0.2)"))
fig_fin.add_trace(go.Scatter(x=df_mensal['mes'], y=df_mensal['receita_bruta'], name='Receita (Linha)', line=dict(color=PREMISSAS['cores']['receita'], width=3), showlegend=False))
fig_fin.add_trace(go.Scatter(x=df_mensal['mes'], y=df_mensal['total_opex'] + df_mensal['total_cogs'], name='Despesas Totais', line=dict(color=PREMISSAS['cores']['despesa'], width=3)))
fig_fin.add_trace(go.Bar(x=df_mensal['mes'], y=df_mensal['lucro_liquido'], name='Lucro/Prejuízo', marker_color=[PREMISSAS['cores']['despesa'] if x < 0 else PREMISSAS['cores']['lucro'] for x in df_mensal['lucro_liquido']]))

# --- ADIÇÃO DE CONTEXTO ---
# Maior Prejuízo
pior_mes_idx = df_mensal['lucro_liquido'].idxmin()
pior_val = df_mensal.loc[pior_mes_idx, 'lucro_liquido']
fig_fin.add_annotation(
    x=df_mensal.loc[pior_mes_idx, 'mes'], y=pior_val,
    text=f"Max Burn: R$ {pior_val:,.0f}",
    showarrow=True, arrowhead=2, ax=0, ay=40, bgcolor="white", bordercolor="red"
)
# Resultado Final
lucro_final = df_mensal['lucro_liquido'].sum()
cor_res = "green" if lucro_final > 0 else "red"
fig_fin.add_annotation(
    x=0.02, y=0.98, xref="paper", yref="paper",
    text=f"<b>RESULTADO ACUMULADO: R$ {lucro_final:,.0f}</b>",
    showarrow=False, bgcolor="white", bordercolor=cor_res
)

fig_fin.update_layout(
    title="<b>💰 Performance Financeira (Mensal)</b>",
    xaxis_title="Mês", yaxis_title="R$", template="plotly_white", height=500, barmode='overlay'
)
fig_fin.show()

# TABELA 2 LIMPA
cols_fin = ['mes', 'receita_bruta', 'total_cogs', 'total_opex', 'ebitda', 'lucro_liquido']
exibir_tabela_grafico(
    df_mensal[df_mensal['mes'].isin([1, 6, 12, 18, 24, 30, 36])],
    {'mes': 'Mês', 'receita_bruta': 'Receita', 'total_cogs': 'COGS', 'total_opex': 'OPEX', 'ebitda': 'EBITDA', 'lucro_liquido': 'Lucro Líq.'},
    "Detalhamento Financeiro (Semestral)"
)

# --- GRÁFICO 3: FLUXO DE CAIXA ---
fig_caixa = make_subplots(specs=[[{"secondary_y": True}]])
fig_caixa.add_trace(go.Scatter(x=df_mensal['mes'], y=df_mensal['caixa'], name='Saldo em Caixa', fill='tozeroy', line=dict(color=PREMISSAS['cores']['caixa'], width=3), fillcolor="rgba(139, 92, 246, 0.1)"), secondary_y=False)
fig_caixa.add_trace(go.Scatter(x=df_mensal['mes'], y=df_mensal['burn_rate'], name='Burn Rate', line=dict(color=PREMISSAS['cores']['danger'], width=2, dash='dot'), mode='lines'), secondary_y=False)
fig_caixa.add_trace(go.Bar(x=df_mensal['mes'], y=df_mensal['aportes_capital'], name='Aportes', marker_color=PREMISSAS['cores']['warning'], opacity=0.5), secondary_y=False)
fig_caixa.add_hline(y=0, line_color="red", line_width=1)

# --- ADIÇÃO DE CONTEXTO ---
min_caixa = df_mensal['caixa'].min()
if min_caixa < 0:
    # Aviso de Quebra
    mes_quebra = df_mensal[df_mensal['caixa'] < 0]['mes'].iloc[0]
    fig_caixa.add_annotation(
        x=mes_quebra, y=0, text=f"💀 QUEBRA (M{mes_quebra})",
        showarrow=True, arrowhead=1, ax=0, ay=-40, bgcolor="red", font=dict(color="white")
    )
    # Necessidade
    fig_caixa.add_annotation(
        x=df_mensal['mes'].iloc[-1], y=min_caixa,
        text=f"Necessidade: R$ {abs(min_caixa):,.0f}",
        showarrow=True, ax=0, ay=40, bgcolor="white", bordercolor="red"
    )

fig_caixa.update_layout(
    title="<b>💸 Fluxo de Caixa & Sobrevivência</b>",
    xaxis_title="Mês", yaxis_title="R$ (Saldo e Burn)", template="plotly_white", height=500, hovermode="x unified"
)
fig_caixa.show()

# TABELA 3 LIMPA
cols_caixa = ['mes', 'caixa', 'burn_rate', 'runway_meses', 'aportes_capital']
exibir_tabela_grafico(
    df_mensal[df_mensal['mes'].isin([1, 6, 12, 18, 24, 30, 36])],
    {'mes': 'Mês', 'caixa': 'Saldo Caixa', 'burn_rate': 'Burn Rate', 'runway_meses': 'Runway (Meses)', 'aportes_capital': 'Aportes'},
    "Evolução do Caixa e Investimentos"
)

# ----------------------------------------------------------------------------
# 4. INSIGHTS
# ----------------------------------------------------------------------------
display(Markdown("### 🧠 Insights Executivos"))
insights = []
insights.append(f"**1. ROI:** {roi_pct:.1f}% sobre investimento de R$ {investimento_total_real:,.2f}.")
if min_caixa < 0:
    insights.append(f"**2. CAIXA CRÍTICO:** O caixa quebra no mês {df_mensal[df_mensal['caixa'] < 0]['mes'].iloc[0]}. Necessidade mínima de aporte adicional: R$ {abs(min_caixa):,.0f}.")
else:
    insights.append(f"**2. CAIXA SEGURO:** O saldo permanece positivo. Mínimo: R$ {min_caixa:,.2f}.")
display(Markdown("\n".join(insights)))
print("\n✅ Célula 06 Concluída (Tabelas Limpas + Contexto Gráfico).")

🧮 AUDITORIA DE CÁLCULOS (ROI & INVESTIMENTO)

🔍 DETALHAMENTO DO CÁLCULO DE ROI:
   (+) Caixa Inicial:          R$ 0.00
   (+) Total Aportes (36m):    R$ 20,000.00
   (=) TOTAL INVESTIDO:        R$ 20,000.00
   --------------------------------------------------
   (+) Lucro Líq. Acumulado:   R$ 54,869.47
   (=) RETORNO TOTAL:          R$ 54,869.47
   --------------------------------------------------
   🎯 ROI FINAL VALIDADO:      274.3%


# 📊 DASHBOARD EXECUTIVO (V7.3 - OTIMIZADO)

Semana,Usuários (Proj),Meta,Gap
1,6,5,1
4,9,5,4
8,14,6,8
13,21,7,14
17,25,8,17
21,30,9,21
26,37,11,26


Mês,Receita,COGS,OPEX,EBITDA,Lucro Líq.
1,719.20,124.40,"2,960.00","-2,472.23","R$ -2,805.57"
6,"3,446.30",541.57,"2,960.00",-568.16,R$ -901.49
12,"10,598.80","1,283.06","4,532.24","3,206.16","R$ 2,872.83"
18,"22,956.00","2,297.80","17,759.28",-517.45,R$ -850.78
24,"38,559.60","3,741.45","23,522.28","5,557.34","R$ 5,224.00"
30,"66,520.40","6,101.68","45,025.20","5,493.78","R$ 5,493.78"
36,"94,651.00","7,894.03","62,391.00","10,279.77","R$ 10,279.77"


Mês,Saldo Caixa,Burn Rate,Runway (Meses),Aportes
1,"R$ -8,472.23","R$ 2,805.57",-3.0,"R$ 2,000.00"
6,"R$ -6,407.70",R$ 901.49,-7.1,"R$ 2,000.00"
12,"R$ 12,073.78",R$ 0.00,999.0,R$ 0.00
18,"R$ 12,781.02",R$ 850.78,15.0,R$ 0.00
24,"R$ 29,028.70",R$ 0.00,999.0,R$ 0.00
30,"R$ 55,880.89",R$ 0.00,999.0,R$ 0.00
36,"R$ 62,869.47",R$ 0.00,999.0,R$ 0.00


### 🧠 Insights Executivos

**1. ROI:** 274.3% sobre investimento de R$ 20,000.00.
**2. CAIXA CRÍTICO:** O caixa quebra no mês 1. Necessidade mínima de aporte adicional: R$ 8,878.


✅ Célula 06 Concluída (Tabelas Limpas + Contexto Gráfico).


In [ ]:
# HOTFIX: Atualiza as cores na memória agora mesmo
CORES.update({
    'primary': '#0F172A',
    'secondary': '#64748B',
    'background': '#FFFFFF',
    'grid': '#F1F5F9',
    'receita': '#10B981',
    'despesa': '#EF4444',
    'lucro': '#3B82F6',
    'ebitda': '#6366F1',
    'caixa': '#8B5CF6',
    'success': '#10B981',
    'danger': '#EF4444',
    'info': '#3B82F6',
    'warning': '#F59E0B',
    'marketing_pago': '#3B82F6',
    'marketing_organico': '#10B981'
})

# Recria a função hex_to_rgba se ela não existir
def hex_to_rgba(hex_color, alpha=0.125):
    hex_color = hex_color.lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r}, {g}, {b}, {alpha})'

print("✅ Cores e funções atualizadas na memória! Tente rodar os gráficos novamente.")

✅ Cores e funções atualizadas na memória! Tente rodar os gráficos novamente.


In [ ]:
# CÉLULA 07 - GROWTH DASHBOARD V8.0 (RESTORATION & FIX)
# ============================================================================
# OBJETIVO: Restaurar gráficos de Growth (Funil, Cohort, Mix).
# CORREÇÃO: Remove dependência de chaves estáticas inexistentes na Célula 02.
# Calcula Cohorts e Funil dinamicamente baseados no df_mensal.
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ----------------------------------------------------------------------------
# SETUP DE CORES
# ----------------------------------------------------------------------------
CORES = PREMISSAS['cores']

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip('#')
    return f"rgba({int(hex_color[0:2], 16)}, {int(hex_color[2:4], 16)}, {int(hex_color[4:6], 16)}, {alpha})"

print("="*80)
print("📈 DASHBOARD DE GROWTH & TRAÇÃO (RESTAURADO)")
print("="*80)

# ============================================================================
# 1. FUNIL DE CONVERSÃO (SANKEY DIAGRAM)
# ============================================================================
print("\n🔀 Gerando Funil de Conversão (Sankey)...\n")

# Pega o último mês com dados reais (ou mês 12 para ter volume)
mes_ref = min(12, len(df_mensal)-1)

# Recupera dados do DataFrame
trafego_pago = df_mensal.loc[mes_ref, 'trafego_pago']
trafego_org = df_mensal.loc[mes_ref, 'trafego_organico']
trials_pagos = df_mensal.loc[mes_ref, 'trials_pagos']
trials_org = df_mensal.loc[mes_ref, 'trials_organicos']
novos_ads = df_mensal.loc[mes_ref, 'novos_ads']
novos_org = df_mensal.loc[mes_ref, 'novos_organicos']
novos_afiliados = df_mensal.loc[mes_ref, 'novos_afiliados']

# Nós do Sankey
labels = [
    f"Tráfego Pago<br>{int(trafego_pago)}",       # 0
    f"Tráfego Orgânico<br>{int(trafego_org)}",    # 1
    f"Trials Pagos<br>{int(trials_pagos)}",       # 2
    f"Trials Orgânicos<br>{int(trials_org)}",     # 3
    f"Clientes Ads<br>{int(novos_ads)}",          # 4
    f"Clientes Org<br>{int(novos_org)}",          # 5
    f"Clientes Afiliados<br>{int(novos_afiliados)}" # 6
]

# Fluxos (Source -> Target)
source = [0, 1, 2, 3, 3] # Pago->TrialP, Org->TrialO, TrialP->Ads, TrialO->Org, TrialO->Afiliado (Simplificação)
target = [2, 3, 4, 5, 6]
values = [
    trials_pagos,       # Pago -> Trial Pago
    trials_org,         # Org -> Trial Org
    novos_ads,          # Trial Pago -> Cliente Ads
    novos_org,          # Trial Org -> Cliente Org
    novos_afiliados     # Trial Org -> Afiliado (Assumindo origem orgânica/viral)
]

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20, line=dict(color="black", width=0.5),
        label=labels, color=[CORES['marketing_pago'], CORES['marketing_organico'], CORES['info'], CORES['info'], CORES['success'], CORES['success'], CORES['warning']]
    ),
    link=dict(source=source, target=target, value=values)
)])

fig_sankey.update_layout(title_text=f"<b>Funil de Conversão (Mês {mes_ref+1})</b>", height=450, font_size=12)
fig_sankey.show()


# ============================================================================
# 2. EVOLUÇÃO DE USUÁRIOS (ATIVOS vs NOVOS vs CHURN)
# ============================================================================
print("\n👥 Gerando Evolução de Usuários...\n")

fig_user = make_subplots(specs=[[{"secondary_y": True}]])

# Área: Usuários Ativos
fig_user.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['usuarios_ativos'],
    name='Base Ativa', fill='tozeroy',
    line=dict(color=CORES['primary'], width=3),
    fillcolor=hex_to_rgba(CORES['primary'], 0.1)
), secondary_y=False)

# Barras: Novos (Verde) e Churn (Vermelho)
fig_user.add_trace(go.Bar(
    x=df_mensal['mes'], y=df_mensal['novos_pagantes_total'],
    name='Novos Clientes', marker_color=CORES['success'], opacity=0.7
), secondary_y=True)

fig_user.add_trace(go.Bar(
    x=df_mensal['mes'], y=-df_mensal['churn_usuarios'],
    name='Cancelamentos', marker_color=CORES['danger'], opacity=0.7
), secondary_y=True)

fig_user.update_layout(title="<b>Dinâmica da Base: Ativos vs Entrada/Saída</b>", template="plotly_white", height=450, hovermode="x unified")
fig_user.show()


# ============================================================================
# 3. MIX DE AQUISIÇÃO (EMPURRADO)
# ============================================================================
print("\n📊 Gerando Mix de Canais...\n")

fig_mix = go.Figure()
fig_mix.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['novos_ads'],
    mode='lines', stackgroup='one', name='Tráfego Pago',
    line=dict(width=0.5, color=CORES['marketing_pago'])
))
fig_mix.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['novos_organicos'],
    mode='lines', stackgroup='one', name='Orgânico/SEO',
    line=dict(width=0.5, color=CORES['marketing_organico'])
))
fig_mix.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['novos_afiliados'],
    mode='lines', stackgroup='one', name='Afiliados',
    line=dict(width=0.5, color=CORES['warning'])
))

fig_mix.update_layout(title="<b>Mix de Aquisição (Canais)</b>", xaxis_title="Mês", yaxis_title="Novos Usuários", template="plotly_white", height=450)
fig_mix.show()


# ============================================================================
# 4. ANÁLISE DE COHORT (RETENÇÃO COM EVOLUÇÃO DE PRODUTO) - CORRIGIDO
# ============================================================================
print("\n🔥 Gerando Cohort Heatmap (Com Evolução de PMF)...\n")

# Parâmetros de Churn da Célula 02
churn_ini = PREMISSAS['churn_inicial']
decaimento = PREMISSAS['churn_decaimento_mensal']
churn_min = PREMISSAS['churn_maturidade']

# Simulação de Cohort para 12 meses
n_meses = 12
z_data = []
y_labels = [f"Safra M{i+1}" for i in range(n_meses)]
x_labels = [f"Mês {i}" for i in range(n_meses)]

# Fator de Melhoria do Produto (Hardcoded lógico para simulação: 1.5% melhor a cada mês)
# Isso faz com que a Safra 12 seja melhor que a Safra 1
fator_melhoria_pmf = 0.015 

for mes_entrada in range(n_meses):
    row = []
    acumulado = 1.0
    
    # Ajusta o Churn Inicial desta safra específica (Melhora com o tempo)
    # A Safra M1 começa com churn_ini (12%)
    # A Safra M10 começa com um churn menor (ex: 10%), pois o produto evoluiu
    churn_start_safra = max(churn_min, churn_ini * (1 - (mes_entrada * fator_melhoria_pmf)))
    
    for mes_vida in range(n_meses):
        if (mes_entrada + mes_vida) >= n_meses:
            row.append(None) # Futuro (Triângulo vazio)
        else:
            if mes_vida == 0:
                row.append(1.0) # Mês 0 é sempre 100%
            else:
                # Calcula o churn deste mês de vida para esta safra específica
                # O churn cai conforme o cliente fica mais velho (decaimento)
                c_atual = max(churn_min, churn_start_safra - (mes_vida * decaimento))
                
                acumulado *= (1 - c_atual)
                row.append(acumulado)
                
    z_data.append(row)

# Inverte para formato triangular clássico (Safra recente em baixo)
z_data = z_data[::-1]
y_labels = y_labels[::-1]

fig_cohort = go.Figure(data=go.Heatmap(
    z=z_data, x=x_labels, y=y_labels,
    colorscale='Blues', # Mantive azul, mas agora os tons vão variar
    text=[[f"{v*100:.0f}%" if v else "" for v in r] for r in z_data],
    texttemplate="%{text}",
    showscale=True
))

fig_cohort.update_layout(
    title="<b>Cohort Analysis (Retenção Evolutiva - PMF)</b><br><span style='font-size:12px;color:gray'>Nota: Safras mais recentes (fundo) retêm melhor devido à evolução do produto.</span>",
    height=500,
    template="plotly_white"
)
fig_cohort.show()

print("\n✅ Célula 07 (Growth) Concluída. Agora as safras evoluem.")

📈 DASHBOARD DE GROWTH & TRAÇÃO (RESTAURADO)

🔀 Gerando Funil de Conversão (Sankey)...




👥 Gerando Evolução de Usuários...




📊 Gerando Mix de Canais...




🔥 Gerando Cohort Heatmap (Com Evolução de PMF)...




✅ Célula 07 (Growth) Concluída. Agora as safras evoluem.


In [ ]:
# CÉLULA 08: DRE & FINANCEIRO (DEMONSTRATIVO DE RESULTADOS)
# ============================================================================
# DEPENDE: df_mensal, df_anual
# ============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

print("="*80)
print("💰 DASHBOARD: DRE & FINANCEIRO")
print("="*80)

# ============================================================================
# 1. DRE WATERFALL (CASCATA MÊS 36)
# ============================================================================
print("\n🌊 Gerando DRE Waterfall (Mês 36)...\n")

mes_ref = 35  # Mês 36 (índice 35)
dados_mes = df_mensal.iloc[mes_ref]

fig_waterfall = go.Figure(go.Waterfall(
    name="DRE",
    orientation="v",
    measure=["relative", "relative", "total", "relative", "relative", "total", "relative", "total"],
    x=["Receita Bruta", "Impostos/Taxas", "Receita Líquida", "COGS (Custo Var.)", "Marketing", "Margem Contrib.", "OPEX (Fixo)", "EBITDA"],
    textposition="outside",
    text=[formatar_moeda(x) for x in [
        dados_mes['receita_bruta'], 
        -dados_mes['total_deducoes'], 
        dados_mes['receita_liquida'],
        -dados_mes['total_cogs'],
        -dados_mes['gasto_marketing'],
        dados_mes['margem_contribuicao'],
        -dados_mes['total_opex'],
        dados_mes['ebitda']
    ]],
    y=[
        dados_mes['receita_bruta'], 
        -dados_mes['total_deducoes'], 
        0, # Total calculado automaticamente
        -dados_mes['total_cogs'],
        -dados_mes['gasto_marketing'],
        0, # Total calculado
        -dados_mes['total_opex'],
        0  # Total calculado
    ],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    decreasing={"marker": {"color": CORES['despesa']}},
    increasing={"marker": {"color": CORES['receita']}},
    totals={"marker": {"color": CORES['lucro']}}
))

fig_waterfall.update_layout(
    title="🌊 DRE Waterfall (Mês 36)",
    showlegend=False,
    height=500,
    template='plotly_white'
)

fig_waterfall.show()

# ============================================================================
# 2. BREAKDOWN DE CUSTOS (DONUT DUPLO)
# ============================================================================
print("\n🍩 Gerando Breakdown de Custos...\n")

fig_donut = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]],
                         subplot_titles=['Custos Variáveis (COGS)', 'Despesas Fixas (OPEX)'])

# COGS
labels_cogs = ['Infra/IA', 'Afiliados', 'Suporte Var.']
values_cogs = [
    dados_mes['custo_ia_total'], 
    dados_mes['comissao_afiliados'],
    dados_mes['custo_suporte_variavel']
]

fig_donut.add_trace(go.Pie(labels=labels_cogs, values=values_cogs, name="COGS", hole=.4),
              1, 1)

# OPEX
labels_opex = ['Pessoal', 'Marketing', 'Infra Fixa', 'Escritório/Admin']
values_opex = [
    dados_mes['custo_pessoal'],
    dados_mes['gasto_marketing'],
    dados_mes['custo_infra_fixo'],
    dados_mes['despesas_admin'] + dados_mes['despesas_escritorio']
]

fig_donut.add_trace(go.Pie(labels=labels_opex, values=values_opex, name="OPEX", hole=.4),
              1, 2)

fig_donut.update_layout(
    title_text="🍩 Composição de Custos (Mês 36)",
    height=400,
    template='plotly_white'
)

fig_donut.show()

# ============================================================================
# 3. EVOLUÇÃO DE MARGENS (MULTI-LINE)
# ============================================================================
print("\n📈 Gerando Evolução de Margens...\n")

fig_margins = go.Figure()

fig_margins.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['margem_bruta_pct'],
    name='Margem Bruta %',
    line=dict(color=CORES['receita'], width=3)
))

fig_margins.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['margem_contribuicao_pct'],
    name='Margem Contribuição %',
    line=dict(color=CORES['info'], width=2, dash='dash')
))

fig_margins.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['ebitda_margin'],
    name='EBITDA Margin %',
    line=dict(color=CORES['lucro'], width=3)
))

fig_margins.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['margem_liquida'],
    name='Margem Líquida %',
    line=dict(color=CORES['caixa'], width=2)
))

fig_margins.update_layout(
    title="📈 Evolução das Margens (%)",
    xaxis_title="Mês",
    yaxis_title="%",
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig_margins.show()

# ============================================================================
# 4. TABELA DRE STYLED (EXPORT-READY)
# ============================================================================
print("\n📋 Gerando Tabela DRE Anual...\n")

# Seleciona colunas para o DRE Anual
cols_dre = ['receita_bruta', 'total_deducoes', 'receita_liquida', 'total_cogs', 'margem_bruta', 
            'gasto_marketing', 'custo_pessoal', 'total_opex', 'ebitda', 'lucro_liquido']

dre_anual = df_anual[['ano'] + cols_dre].copy()

# Formata
for col in cols_dre:
    dre_anual[col] = dre_anual[col].apply(lambda x: formatar_moeda(x))

# Renomeia colunas
mapa_cols = {
    'receita_bruta': 'Receita Bruta',
    'total_deducoes': '(-) Deduções',
    'receita_liquida': 'Receita Líquida',
    'total_cogs': '(-) COGS',
    'margem_bruta': 'Margem Bruta',
    'gasto_marketing': '(-) Marketing',
    'custo_pessoal': '(-) Pessoal',
    'total_opex': '(-) Total OPEX',
    'ebitda': 'EBITDA',
    'lucro_liquido': 'Lucro Líquido'
}
dre_anual.rename(columns=mapa_cols, inplace=True)

print(dre_anual.T.to_string())


💰 DASHBOARD: DRE & FINANCEIRO

🌊 Gerando DRE Waterfall (Mês 36)...




🍩 Gerando Breakdown de Custos...



KeyError: 'despesas_admin'

In [ ]:
# CÉLULA 09: UNIT ECONOMICS (LTV, CAC, PAYBACK)
# ============================================================================
# DEPENDE: df_mensal
# ============================================================================

print("="*80)
print("📊 DASHBOARD: UNIT ECONOMICS")
print("="*80)

# ============================================================================
# 1. LTV vs CAC (DUAL LINE)
# ============================================================================
print("\n📈 Gerando LTV vs CAC...\n")

fig_ltv_cac = go.Figure()

fig_ltv_cac.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['ltv'],
    name='LTV (Lifetime Value)',
    line=dict(color=CORES['receita'], width=3)
))

fig_ltv_cac.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['cac_blended'],
    name='CAC Blended',
    line=dict(color=CORES['despesa'], width=3)
))

fig_ltv_cac.update_layout(
    title="📈 LTV vs CAC (Evolução)",
    xaxis_title="Mês",
    yaxis_title="R$",
    height=500,
    template='plotly_white',
    hovermode='x unified'
)

fig_ltv_cac.show()

# ============================================================================
# 2. LTV/CAC RATIO (GAUGE)
# ============================================================================
print("\n🧭 Gerando Gauge LTV/CAC...\n")

ltv_cac_atual = df_mensal['ltv_cac'].iloc[-1]

fig_gauge = go.Figure(go.Indicator(
    mode = "gauge+number+delta",
    value = ltv_cac_atual,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': "LTV/CAC Ratio (Mês 36)"},
    delta = {'reference': 3.0, 'increasing': {'color': CORES['success']}},
    gauge = {
        'axis': {'range': [None, 10], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': CORES['primary']},
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 1], 'color': CORES['danger']},
            {'range': [1, 3], 'color': CORES['warning']},
            {'range': [3, 10], 'color': CORES['success']}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 3.0
        }
    }
))

fig_gauge.update_layout(height=400)
fig_gauge.show()

# ============================================================================
# 3. PAYBACK PERIOD (BAR)
# ============================================================================
print("\n⏳ Gerando Payback Period...\n")

fig_payback = go.Figure()

fig_payback.add_trace(go.Bar(
    x=df_mensal['mes'],
    y=df_mensal['payback_meses'],
    name='Payback (Meses)',
    marker_color=CORES['info']
))

fig_payback.add_hline(
    y=12,
    line_dash="dash",
    line_color=CORES['warning'],
    annotation_text="Max 12 Meses"
)

fig_payback.update_layout(
    title="⏳ Payback Period (Meses)",
    xaxis_title="Mês",
    yaxis_title="Meses",
    height=400,
    template='plotly_white'
)

fig_payback.show()

# ============================================================================
# 4. MAGIC NUMBER & REGRA 40
# ============================================================================
print("\n✨ Gerando Magic Number & Regra dos 40...\n")

regra_40_atual = df_mensal['regra_40'].iloc[-1]
status_40 = "✅" if regra_40_atual >= 40 else "❌"

print(f"   MAGIC NUMBER (Eficiência de Vendas): {df_mensal['net_new_mrr'].iloc[-1] / df_mensal['gasto_marketing'].iloc[-2] if df_mensal['gasto_marketing'].iloc[-2] > 0 else 0:.2f}")
print(f"   REGRA DOS 40: {regra_40_atual:.1f}% {status_40} (Meta: 40%)")


📊 DASHBOARD: UNIT ECONOMICS

📈 Gerando LTV vs CAC...




🧭 Gerando Gauge LTV/CAC...




⏳ Gerando Payback Period...




✨ Gerando Magic Number & Regra dos 40...

   MAGIC NUMBER (Eficiência de Vendas): 0.00
   REGRA DOS 40: 0.0% ❌ (Meta: 40%)


In [ ]:
# CÉLULA 10: RISK & PRODUCT (MONTE CARLO + ENGAGEMENT)
# ============================================================================
# DEPENDE: mc_resultados, df_mensal
# ============================================================================

print("="*80)
print("🎲 DASHBOARD: RISK & PRODUCT")
print("="*80)

# ============================================================================
# 1. MONTE CARLO FAN CHART (VISUALIZAÇÃO DE INCERTEZA)
# ============================================================================
print("\n🌪️ Gerando Monte Carlo Fan Chart...\n")

if 'caixa_mensal_p10' in mc_resultados:
    fig_fan = go.Figure()
    
    x_meses = list(range(1, len(mc_resultados['caixa_mensal_p50']) + 1))
    
    # Área P10-P90 (Incerteza)
    fig_fan.add_trace(go.Scatter(
        x=x_meses + x_meses[::-1],
        y=list(mc_resultados['caixa_mensal_p90']) + list(mc_resultados['caixa_mensal_p10'])[::-1],
        fill='toself',
        fillcolor=hex_to_rgba(CORES['primary'], 0.1),
        line=dict(color='rgba(255,255,255,0)'),
        name='Intervalo P10-P90'
    ))
    
    # Mediana P50
    fig_fan.add_trace(go.Scatter(
        x=x_meses,
        y=mc_resultados['caixa_mensal_p50'],
        line=dict(color=CORES['primary'], width=3),
        name='Cenário Base (P50)'
    ))
    
    # Linha Zero
    fig_fan.add_hline(y=0, line_color=CORES['danger'], line_width=2, line_dash="dash", annotation_text="Caixa Zero")
    
    fig_fan.update_layout(
        title="🌪️ Monte Carlo Fan Chart: Projeção de Caixa (Incerteza)",
        xaxis_title="Mês",
        yaxis_title="Caixa Acumulado (R$)",
        height=500,
        template='plotly_white'
    )
    
    fig_fan.show()
else:
    print("⚠️ Dados de Monte Carlo incompletos para Fan Chart.")

# ============================================================================
# 2. HISTOGRAMA DE DISTRIBUIÇÃO (CAIXA FINAL)
# ============================================================================
print("\n📊 Gerando Histograma de Distribuição...\n")

if 'caixa_array' in mc_resultados:
    fig_hist = go.Figure(go.Histogram(
        x=mc_resultados['caixa_array'],
        nbinsx=50,
        marker_color=CORES['caixa'],
        opacity=0.7
    ))
    
    fig_hist.add_vline(x=0, line_color=CORES['danger'], line_width=3, annotation_text="Quebra de Caixa")
    fig_hist.add_vline(x=mc_resultados['caixa_p50'], line_color=CORES['primary'], line_dash="dash", annotation_text="Mediana")
    
    fig_hist.update_layout(
        title="📊 Distribuição de Probabilidade: Caixa no Mês 36",
        xaxis_title="Caixa Final (R$)",
        yaxis_title="Frequência (Simulações)",
        height=400,
        template='plotly_white'
    )
    
    fig_hist.show()

# ============================================================================
# 3. BURN RATE & RUNWAY
# ============================================================================
print("\n🔥 Gerando Burn Rate & Runway...\n")

fig_burn = make_subplots(specs=[[{"secondary_y": True}]])

fig_burn.add_trace(go.Bar(
    x=df_mensal['mes'],
    y=df_mensal['burn_rate'],
    name='Burn Rate (R$)',
    marker_color=CORES['danger'],
    opacity=0.6
), secondary_y=False)

fig_burn.add_trace(go.Scatter(
    x=df_mensal['mes'],
    y=df_mensal['runway_meses'],
    name='Runway (Meses)',
    line=dict(color=CORES['warning'], width=3),
    mode='lines+markers'
), secondary_y=True)

fig_burn.update_layout(
    title="🔥 Burn Rate vs Runway",
    xaxis_title="Mês",
    height=400,
    template='plotly_white'
)

fig_burn.update_yaxes(title_text="Burn Rate (R$)", secondary_y=False)
fig_burn.update_yaxes(title_text="Runway (Meses)", range=[0, 36], secondary_y=True)

fig_burn.show()

# ============================================================================
# 4. PRODUCT ENGAGEMENT (DAU/MAU & NPS)
# ============================================================================
print("\n📱 Gerando Métricas de Produto...\n")

# Simula dados de produto (se não existirem no df)
if 'dau_mau' not in df_mensal.columns:
    # Cria curva simulada baseada nas premissas
    df_mensal['dau_mau'] = np.linspace(PREMISSAS['dau_mau_ratio_inicial'], PREMISSAS['dau_mau_ratio_target'], len(df_mensal))
    df_mensal['nps'] = np.linspace(PREMISSAS['nps_inicial'], PREMISSAS['nps_target'], len(df_mensal))

fig_prod = make_subplots(rows=1, cols=2, subplot_titles=['DAU/MAU Ratio (Stickiness)', 'NPS Evolution'])

# DAU/MAU
fig_prod.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['dau_mau'],
    name='DAU/MAU',
    line=dict(color=CORES['info'], width=3),
    fill='tozeroy'
), row=1, col=1)

# NPS
fig_prod.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['nps'],
    name='NPS',
    line=dict(color=CORES['success'], width=3),
    mode='lines+markers'
), row=1, col=2)

fig_prod.update_layout(height=400, title="📱 Product Engagement Metrics", template='plotly_white')
fig_prod.show()


🎲 DASHBOARD: RISK & PRODUCT

🌪️ Gerando Monte Carlo Fan Chart...




📊 Gerando Histograma de Distribuição...




🔥 Gerando Burn Rate & Runway...




📱 Gerando Métricas de Produto...



KeyError: 'dau_mau_ratio_inicial'

In [ ]:
# CÉLULA 11: OPERATIONS (HEADCOUNT & INFRA)
# ============================================================================
# DEPENDE: df_mensal
# ============================================================================

print("="*80)
print("⚙️ DASHBOARD: OPERATIONS")
print("="*80)

# ============================================================================
# 1. HEADCOUNT EVOLUTION (STACKED AREA)
# ============================================================================
print("\n👥 Gerando Evolução de Headcount...\n")

fig_hc = go.Figure()

fig_hc.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['headcount_fundadores'],
    name='Fundadores',
    stackgroup='one',
    mode='none'
))

fig_hc.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['headcount_dev'],
    name='Engenharia',
    stackgroup='one',
    mode='none'
))

fig_hc.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['headcount_cs'],
    name='Customer Success',
    stackgroup='one',
    mode='none'
))

fig_hc.update_layout(
    title="👥 Evolução do Time (Headcount)",
    xaxis_title="Mês",
    yaxis_title="Pessoas",
    height=400,
    template='plotly_white'
)

fig_hc.show()

# ============================================================================
# 2. REVENUE PER EMPLOYEE (LINE)
# ============================================================================
print("\n💼 Gerando Revenue per Employee...\n")

# Calcula Rev/Emp anualizado
df_mensal['rev_per_emp'] = (df_mensal['arr'] / df_mensal['headcount_total']).replace([np.inf, -np.inf], 0)

fig_rpe = go.Figure()

fig_rpe.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['rev_per_emp'],
    name='ARR / Funcionário',
    line=dict(color=CORES['receita'], width=3)
))

fig_rpe.add_hline(
    y=PREMISSAS['revenue_per_employee_meta'],
    line_dash="dash",
    line_color=CORES['success'],
    annotation_text=f"Meta: {formatar_moeda(PREMISSAS['revenue_per_employee_meta'])}"
)

fig_rpe.update_layout(
    title="💼 Eficiência: ARR por Funcionário",
    xaxis_title="Mês",
    yaxis_title="R$",
    height=400,
    template='plotly_white'
)

fig_rpe.show()

# ============================================================================
# 3. INFRA TIER PROGRESSION
# ============================================================================
print("\n🖥️ Gerando Evolução de Infraestrutura...\n")

fig_infra = make_subplots(specs=[[{"secondary_y": True}]])

fig_infra.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['custo_infra_fixo'],
    name='Custo Infra (R$)',
    line=dict(color=CORES['infra'], width=3),
    fill='tozeroy'
), secondary_y=False)

fig_infra.add_trace(go.Scatter(
    x=df_mensal['mes'], y=df_mensal['infra_tier_ativo'],
    name='Tier Ativo',
    mode='lines+markers',
    line=dict(color=CORES['secondary'], dash='dot')
), secondary_y=True)

fig_infra.update_layout(
    title="🖥️ Custo de Infraestrutura & Tiers",
    height=400,
    template='plotly_white'
)

fig_infra.update_yaxes(title_text="Custo Mensal (R$)", secondary_y=False)
fig_infra.update_yaxes(title_text="Tier", secondary_y=True, dtick=1)

fig_infra.show()


⚙️ DASHBOARD: OPERATIONS

👥 Gerando Evolução de Headcount...




💼 Gerando Revenue per Employee...



KeyError: 'revenue_per_employee_meta'

In [ ]:
# ============================================================================
# CÉLULA 12: APPENDIX & EXPORTS
# ============================================================================
# ============================================================================

import base64
from IPython.display import HTML

def create_download_link(df, title = "Download CSV", filename = "data.csv"):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    payload = f'data:text/csv;base64,{b64}'
    html = '<a download="{filename}" href="{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

print("="*80)
print("📥 DOWNLOADS & EXPORTAÇÃO")
print("="*80)

print("\n1. Dados Mensais Completos (36 Meses)")
display(create_download_link(df_mensal, title="📥 Baixar df_mensal.csv", filename="projecao_mensal_v6.csv"))

print("\n2. DRE Anual Consolidado")
display(create_download_link(df_anual, title="📥 Baixar df_anual.csv", filename="dre_anual_v6.csv"))

print("\n" + "="*80)
print("✅ PROJETO FINALIZADO COM SUCESSO!")
print("="*80)
